# Governance-Aware Evidence Fusion and Decision Engine

This notebook implements the calibrated evidence-fusion policy and its governed routing layer. It consumes the validated baseline artifacts, performs nested experiment-level calibration, evaluates detection and routing operating points, computes statistical and operational comparisons, and generates publication artifacts.

**Policy version:** `HYBRID_ROUTING_POLICY_V3.0_UNIFIED`

## Execution order

Run `01_Baseline_Prototype.ipynb` first. This notebook verifies all required upstream artifacts before analysis.

## Unified policy

The final routing branch uses the calibrated fused risk score. Deterministic rule severity and evidence conflict retain lexicographic precedence, producing one policy with a detection operating point and a full governed-routing operating point.


In [ ]:
# ================================================================
# Environment setup, Drive paths, and baseline artifact validation
# ================================================================

import gc
import json
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

from IPython.display import display

try:
    from google.colab import drive
    IN_GOOGLE_COLAB = True
except ImportError:
    drive = None
    IN_GOOGLE_COLAB = False


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


# ----------------------------------------------------------------
# Project paths — Google Drive
# ----------------------------------------------------------------
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Paper2_ZTLF")

if USE_GOOGLE_DRIVE:
    if not IN_GOOGLE_COLAB:
        raise RuntimeError("USE_GOOGLE_DRIVE=True requires a Google Colab runtime.")
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = GOOGLE_DRIVE_PROJECT_ROOT
else:
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir

DATA_DIR       = PROJECT_ROOT / "data"
BRONZE_DIR     = DATA_DIR / "bronze"
EXPERIMENT_DIR = DATA_DIR / "experiments"
RESULTS_DIR    = PROJECT_ROOT / "results"
MODEL_DIR      = PROJECT_ROOT / "models"
FIGURES_DIR    = PROJECT_ROOT / "figures"
TABLES_DIR     = PROJECT_ROOT / "tables"

MANIFEST_DIR    = RESULTS_DIR / "sample_manifests"
RULE_BY_EXP_DIR = RESULTS_DIR / "rule_results_by_experiment"
AI_RESULT_DIR   = RESULTS_DIR / "ai_results_by_experiment"

HYBRID_ROOT        = RESULTS_DIR / "hybrid_decision_engine"
HYBRID_DATA_DIR    = HYBRID_ROOT / "data"
HYBRID_RESULTS_DIR = HYBRID_ROOT / "results"
HYBRID_AUDIT_DIR   = HYBRID_ROOT / "audit"
HYBRID_CONFIG_DIR  = PROJECT_ROOT / "configs"

for directory in [HYBRID_ROOT, HYBRID_DATA_DIR, HYBRID_RESULTS_DIR,
                  HYBRID_AUDIT_DIR, HYBRID_CONFIG_DIR, FIGURES_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def memory_report(label=""):
    gc.collect()
    rss_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    available_gb = psutil.virtual_memory().available / (1024 ** 3)
    print(f"[MEM] {label:<38} used={rss_gb:5.2f} GB   available={available_gb:5.2f} GB")
    return rss_gb


# ----------------------------------------------------------------
# Baseline artifact validation
# ----------------------------------------------------------------
experiment_registry_df = pd.read_csv(RESULTS_DIR / "experiment_registry.csv")
EXPERIMENT_COUNT = len(experiment_registry_df)

manifest_files = sorted(MANIFEST_DIR.glob("*_manifest.parquet"))
rule_files     = sorted(RULE_BY_EXP_DIR.glob("*_rule_records.parquet"))
ai_files       = sorted(AI_RESULT_DIR.glob("*_ai_results.parquet"))

validation = pd.DataFrame([
    {"artifact": "experiment_registry.csv",     "found": EXPERIMENT_COUNT, "expected": 63},
    {"artifact": "sample_manifests/",           "found": len(manifest_files), "expected": 63},
    {"artifact": "rule_results_by_experiment/", "found": len(rule_files), "expected": 63},
    {"artifact": "ai_results_by_experiment/",   "found": len(ai_files), "expected": 63},
])
validation["ok"] = validation["found"] == validation["expected"]
display(validation)

if not validation["ok"].all():
    raise FileNotFoundError(
        "Baseline artifacts are incomplete or come from an earlier notebook version.\n"
        "Run 01_Baseline_Prototype_MEMSAFE.ipynb to completion first.\n"
        "If sample_manifests/ or rule_results_by_experiment/ is missing entirely, "
        "the baseline that produced these outputs predates the manifest design."
    )


# ----------------------------------------------------------------
# Staleness guard
# ----------------------------------------------------------------
# regenerated. Detect that class of problem before doing any work.
manifest_summary_df = pd.read_csv(RESULTS_DIR / "sample_manifest_summary.csv")

prevalence_drift = (
    manifest_summary_df["realized_prevalence"] - manifest_summary_df["anomaly_rate"]
).abs()
if (prevalence_drift >= 0.005).any():
    raise ValueError(
        "Manifest prevalence does not match the nominal injection rate:\n"
        f"{manifest_summary_df.loc[prevalence_drift >= 0.005]}"
    )

if (manifest_summary_df["negative_records"] < 100).any():
    raise ValueError(
        "Degenerate experiments detected (fewer than 100 negative records). "
        "These make FPR and specificity undefined."
    )

_probe = pd.read_parquet(ai_files[0])
_required_columns = {"experiment_id", "dataset", "anomaly_type", "anomaly_rate",
                     "record_id", "detector", "anomaly_score", "anomaly_prediction"}
_missing = _required_columns - set(_probe.columns)
if _missing:
    raise ValueError(
        f"AI result files are missing columns: {sorted(_missing)}.\n"
        "This indicates output from an earlier scoring cell. Re-run the baseline "
        "with PURGE_STALE_ARTIFACTS = True."
    )
SAMPLE_RECORDS_PER_EXPERIMENT = int(len(_probe) / _probe["detector"].nunique())
del _probe
gc.collect()


# ----------------------------------------------------------------
# Run metadata
# ----------------------------------------------------------------
HYBRID_RUN_ID = "HYBRID_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

with open(HYBRID_AUDIT_DIR / "hybrid_execution_metadata.json", "w") as handle:
    json.dump({
        "hybrid_run_id": HYBRID_RUN_ID,
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "random_seed": RANDOM_SEED,
        "project_root": str(PROJECT_ROOT),
        "experiments": EXPERIMENT_COUNT,
        "records_per_experiment": SAMPLE_RECORDS_PER_EXPERIMENT,
    }, handle, indent=2)

print("-" * 78)
print(f"Hybrid run ID           : {HYBRID_RUN_ID}")
print(f"Project root            : {PROJECT_ROOT}")
print(f"Experiments             : {EXPERIMENT_COUNT}")
print(f"Records per experiment  : {SAMPLE_RECORDS_PER_EXPERIMENT:,}")
print(f"Expected fused rows     : {EXPERIMENT_COUNT * SAMPLE_RECORDS_PER_EXPERIMENT:,}")
print("-" * 78)
memory_report("after setup")


## 2. Build the Record-Level Evidence Table

Evidence is assembled one experiment at a time. Manifest labels, deterministic rule evidence, and both detector outputs are joined on `(experiment_id, record_id)` and validated one-to-one.

The manifest ensures that all evidence sources cover the same sampled records and prevents silent row duplication or loss during joins.


In [ ]:
# ================================================================
# Assemble the record-level evidence table, one experiment at a time
# ================================================================

import gc
import time

import numpy as np
import pandas as pd


assembly_started = time.perf_counter()
evidence_frames = []

for position, experiment in experiment_registry_df.iterrows():
    experiment_id = experiment["experiment_id"]
    dataset       = experiment["dataset"]

    # -- manifest: record ids and ground truth ---------------------
    manifest = pd.read_parquet(MANIFEST_DIR / f"{experiment_id}_manifest.parquet")

    # -- detector evidence: long -> wide ---------------------------
    scores = pd.read_parquet(
        AI_RESULT_DIR / f"{experiment_id}_ai_results.parquet",
        columns=["record_id", "detector", "anomaly_score", "anomaly_prediction"],
    )

    # Rank-normalize within experiment and detector so the two detectors
    # are on a common scale before fusion.
    scores["normalized_score"] = (
        scores.groupby("detector")["anomaly_score"]
        .rank(method="average", pct=True)
        .fillna(0.0)
        .clip(0.0, 1.0)
    )

    score_wide = scores.pivot_table(
        index="record_id", columns="detector",
        values="normalized_score", aggfunc="max",
    ).rename(columns={"ISOLATION_FOREST": "iforest_normalized_score",
                      "LOCAL_OUTLIER_FACTOR": "lof_normalized_score"})
    prediction_wide = scores.pivot_table(
        index="record_id", columns="detector",
        values="anomaly_prediction", aggfunc="max",
    ).rename(columns={"ISOLATION_FOREST": "iforest_prediction",
                      "LOCAL_OUTLIER_FACTOR": "lof_prediction"})

    detector_df = score_wide.join(prediction_wide, how="outer").reset_index()
    detector_df.columns.name = None
    del scores, score_wide, prediction_wide

    # -- deterministic rule evidence -------------------------------
    rules = pd.read_parquet(
        RULE_BY_EXP_DIR / f"{experiment_id}_rule_records.parquet",
        columns=["record_id", "failed_rule_count", "trust_score",
                 "max_severity_score", "rule_status", "rule_risk_score"],
    )

    # -- join ------------------------------------------------------
    evidence = (
        manifest
        .merge(detector_df, on="record_id", how="left", validate="one_to_one")
        .merge(rules, on="record_id", how="left", validate="one_to_one")
    )
    evidence["anomaly_type"] = experiment["anomaly_type"]
    evidence["anomaly_rate"] = experiment["anomaly_rate"]

    assert len(evidence) == len(manifest), \
        f"{experiment_id}: join changed row count"

    evidence_frames.append(evidence)
    del manifest, detector_df, rules, evidence
    gc.collect()

    if (position + 1) % 20 == 0:
        memory_report(f"assembly {position + 1}/{EXPERIMENT_COUNT}")

hybrid_base_df = pd.concat(evidence_frames, ignore_index=True)
del evidence_frames
gc.collect()

print(f"Assembly runtime: {time.perf_counter() - assembly_started:.1f}s")
print(f"Evidence rows: {len(hybrid_base_df):,}")


# ----------------------------------------------------------------
# Derived evidence terms
# ----------------------------------------------------------------
hybrid_base_df["rule_evidence_available"] = (
    hybrid_base_df["rule_risk_score"].notna().astype(int)
)
hybrid_base_df["detector_evidence_available"] = (
    hybrid_base_df[["iforest_normalized_score", "lof_normalized_score"]]
    .notna().all(axis=1).astype(int)
)

for column, default in [("rule_risk_score", 0.0), ("max_severity_score", 0.0),
                        ("failed_rule_count", 0.0), ("trust_score", 1.0),
                        ("iforest_normalized_score", 0.0), ("lof_normalized_score", 0.0),
                        ("iforest_prediction", 0), ("lof_prediction", 0)]:
    hybrid_base_df[column] = hybrid_base_df[column].fillna(default)

hybrid_base_df = hybrid_base_df.rename(columns={
    "max_severity_score": "rule_severity_score",
    "trust_score": "rule_trust_score",
})

# Detector agreement, expected downstream as the complement of disagreement.
hybrid_base_df["ai_detector_agreement"] = (
    1.0 - (hybrid_base_df["iforest_normalized_score"]
           - hybrid_base_df["lof_normalized_score"]).abs()
).clip(0.0, 1.0)

hybrid_base_df["rule_anomaly_prediction"] = (
    hybrid_base_df["rule_status"].eq("FAIL").astype(int)
)

# Detector consensus, vote, disagreement.
hybrid_base_df["ai_consensus_score"] = hybrid_base_df[
    ["iforest_normalized_score", "lof_normalized_score"]
].mean(axis=1)

hybrid_base_df["ai_prediction_vote"] = hybrid_base_df[
    ["iforest_prediction", "lof_prediction"]
].mean(axis=1)

hybrid_base_df["ai_score_disagreement"] = (
    hybrid_base_df["iforest_normalized_score"]
    - hybrid_base_df["lof_normalized_score"]
).abs()

prediction_disagreement = (
    hybrid_base_df["iforest_prediction"] != hybrid_base_df["lof_prediction"]
).astype(float)

# Evidence completeness: rule evidence, iforest, lof — three sources.
hybrid_base_df["evidence_completeness"] = (
    hybrid_base_df["rule_evidence_available"]
    + hybrid_base_df["detector_evidence_available"] * 2
) / 3.0

evidence_missingness = 1.0 - hybrid_base_df["evidence_completeness"]

hybrid_base_df["evidence_uncertainty_score"] = (
    0.40 * evidence_missingness
    + 0.35 * hybrid_base_df["ai_score_disagreement"]
    + 0.25 * prediction_disagreement
).clip(0.0, 1.0)

# ----------------------------------------------------------------
# Column contract expected by the downstream policy cells
# ----------------------------------------------------------------
REQUIRED_EVIDENCE_COLUMNS = [
    "experiment_id", "dataset", "record_id", "ground_truth_label",
    "anomaly_type", "anomaly_rate",
    "rule_risk_score", "rule_trust_score", "rule_severity_score",
    "rule_anomaly_prediction", "rule_evidence_available",
    "iforest_normalized_score", "lof_normalized_score",
    "iforest_prediction", "lof_prediction",
    "ai_consensus_score", "ai_prediction_vote", "ai_detector_agreement",
    "ai_score_disagreement", "evidence_completeness",
    "evidence_uncertainty_score",
]
_missing_contract = [c for c in REQUIRED_EVIDENCE_COLUMNS
                     if c not in hybrid_base_df.columns]
assert not _missing_contract, f"evidence table missing: {_missing_contract}"

hybrid_base_df.to_parquet(
    HYBRID_DATA_DIR / "hybrid_record_level_input.parquet", index=False
)

# ----------------------------------------------------------------
# Integrity checks
# ----------------------------------------------------------------
assert len(hybrid_base_df) == EXPERIMENT_COUNT * SAMPLE_RECORDS_PER_EXPERIMENT, (
    f"expected {EXPERIMENT_COUNT * SAMPLE_RECORDS_PER_EXPERIMENT:,} rows, "
    f"got {len(hybrid_base_df):,}"
)
assert hybrid_base_df.duplicated(subset=["experiment_id", "record_id"]).sum() == 0, \
    "duplicate (experiment_id, record_id) pairs — the join fanned out"
assert hybrid_base_df["rule_evidence_available"].mean() > 0.99, \
    "rule evidence missing for a large share of records"

# was identically zero because rules had been evaluated on clean data.
nonzero_rule_risk = (hybrid_base_df["rule_risk_score"] > 0).sum()
assert nonzero_rule_risk > 0, (
    "rule_risk_score is identically zero. The rule engine contributed nothing, "
    "which means it never saw the injected anomalies."
)

print(f"\nRecords with non-zero rule risk : {nonzero_rule_risk:,} "
      f"({nonzero_rule_risk / len(hybrid_base_df):.2%})")
print(f"Mean detector consensus         : {hybrid_base_df['ai_consensus_score'].mean():.4f}")
print(f"Mean evidence completeness      : {hybrid_base_df['evidence_completeness'].mean():.4f}")
print(f"Overall anomaly prevalence      : {hybrid_base_df['ground_truth_label'].mean():.4f}")
print("\nPASS: evidence table assembled and validated.")
memory_report("after assembly")


## 4. Hybrid Risk Scoring and Governed Decisions

This section applies the explicit policy-based fusion layer and maps combined evidence into auditable operational decisions.


In [ ]:
# ======================================================================
# Section 4 — Decision policies
#
# TWO policies are defined here and must never be conflated:
#
#   assign_hybrid_decision()          the UNIFIED governed router. Its final
#                                     branch uses the calibrated fused risk
#                                     R(i) >= threshold, both read from the
#                                     nested-CV artifact.
#
#   assign_v1_prototype_decision()    the FROZEN uncalibrated prototype. Its
#                                     final branch uses raw detector consensus
#                                     >= 0.90, the original hand-set cutoff.
#
# The calibrated router and frozen uncalibrated prototype are maintained
# as separate methods so that RQ2 can be evaluated without label ambiguity.
# ======================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

cell_started = time.perf_counter()

# ----------------------------------------------------------------
# Calibrated configuration, read from the artifact. Never hardcoded.
# ----------------------------------------------------------------
selected_config = pd.read_csv(
    HYBRID_RESULTS_DIR / "hybrid_v3_selected_configurations.csv"
)
assert selected_config["threshold"].nunique() == 1, \
    "outer folds selected different thresholds — policy is not stable"
assert selected_config[[
    "rule_risk_weight", "ai_consensus_weight",
    "ai_prediction_vote_weight", "uncertainty_weight",
]].nunique().eq(1).all(), \
    "outer folds selected different weights — policy is not stable"

RULE_WEIGHT        = float(selected_config["rule_risk_weight"].iloc[0])
CONSENSUS_WEIGHT   = float(selected_config["ai_consensus_weight"].iloc[0])
VOTE_WEIGHT        = float(selected_config["ai_prediction_vote_weight"].iloc[0])
UNCERTAINTY_WEIGHT = float(selected_config["uncertainty_weight"].iloc[0])
FUSED_THRESHOLD    = float(selected_config["threshold"].iloc[0])

ROUTING_POLICY = {
    "policy_version": "HYBRID_ROUTING_POLICY_V3.0_UNIFIED",
    "critical_rule_severity": 0.75,
    "uncertainty_escalation": 0.50,
    "minimum_evidence_completeness": 2.0 / 3.0,
    "fused_risk_weights": {
        "rule_risk": RULE_WEIGHT,
        "ai_consensus": CONSENSUS_WEIGHT,
        "ai_prediction_vote": VOTE_WEIGHT,
        "uncertainty": UNCERTAINTY_WEIGHT,
    },
    "fused_risk_threshold": FUSED_THRESHOLD,
    "precedence": [
        "CRITICAL rule violation   -> QUARANTINE",
        "Any rule violation        -> REPAIR",
        "Incomplete evidence       -> ESCALATE",
        "High uncertainty          -> ESCALATE",
        "Fused risk >= threshold   -> QUARANTINE",
        "Otherwise                 -> ACCEPT",
    ],
}

V1_PROTOTYPE_POLICY = {
    "policy_version": "HYBRID_POLICY_V1_PROTOTYPE_FROZEN",
    "critical_rule_severity": 0.75,
    "uncertainty_escalation": 0.50,
    "minimum_evidence_completeness": 2.0 / 3.0,
    "detector_quarantine_threshold": 0.90,
    "note": "Uncalibrated hand-set prototype. Must not track the router.",
}

print("Unified router bound to calibrated configuration:")
print(f"  weights   rule={RULE_WEIGHT}  consensus={CONSENSUS_WEIGHT}  "
      f"vote={VOTE_WEIGHT}  uncertainty={UNCERTAINTY_WEIGHT}")
print(f"  threshold {FUSED_THRESHOLD}")
print(f"V1 prototype detector cutoff: "
      f"{V1_PROTOTYPE_POLICY['detector_quarantine_threshold']} (raw consensus)")


# ----------------------------------------------------------------
# Decision functions
# ----------------------------------------------------------------
def fused_risk(row) -> float:
    """Calibrated fused risk R(i). Weights come from nested CV."""
    return (
        RULE_WEIGHT        * float(row["rule_risk_score"])
        + CONSENSUS_WEIGHT * float(row["ai_consensus_score"])
        + VOTE_WEIGHT      * float(row["ai_prediction_vote"])
        + UNCERTAINTY_WEIGHT * float(row["evidence_uncertainty_score"])
    )


def assign_hybrid_decision(row) -> str:
    """Unified governed router. Lexicographic precedence."""
    if float(row["rule_severity_score"]) >= ROUTING_POLICY["critical_rule_severity"]:
        return "QUARANTINE"
    if int(row["rule_anomaly_prediction"]) == 1:
        return "REPAIR"
    if float(row["evidence_completeness"]) < ROUTING_POLICY["minimum_evidence_completeness"]:
        return "ESCALATE"
    if float(row["evidence_uncertainty_score"]) > ROUTING_POLICY["uncertainty_escalation"]:
        return "ESCALATE"
    return "QUARANTINE" if fused_risk(row) >= FUSED_THRESHOLD else "ACCEPT"


def assign_v1_prototype_decision(row) -> str:
    """Frozen uncalibrated prototype. Identical precedence, uncalibrated cutoff."""
    if float(row["rule_severity_score"]) >= V1_PROTOTYPE_POLICY["critical_rule_severity"]:
        return "QUARANTINE"
    if int(row["rule_anomaly_prediction"]) == 1:
        return "REPAIR"
    if float(row["evidence_completeness"]) < V1_PROTOTYPE_POLICY["minimum_evidence_completeness"]:
        return "ESCALATE"
    if float(row["evidence_uncertainty_score"]) > V1_PROTOTYPE_POLICY["uncertainty_escalation"]:
        return "ESCALATE"
    if float(row["ai_consensus_score"]) >= V1_PROTOTYPE_POLICY["detector_quarantine_threshold"]:
        return "QUARANTINE"
    return "ACCEPT"


# ----------------------------------------------------------------
# Apply both policies
# ----------------------------------------------------------------
hybrid_decision_df = hybrid_base_df.copy()

hybrid_decision_df["hybrid_risk_score"]  = hybrid_decision_df.apply(fused_risk, axis=1)
hybrid_decision_df["hybrid_trust_score"] = 1.0 - hybrid_decision_df["hybrid_risk_score"]

hybrid_decision_df["hybrid_decision"] = hybrid_decision_df.apply(
    assign_hybrid_decision, axis=1)
hybrid_decision_df["v1_prototype_decision"] = hybrid_decision_df.apply(
    assign_v1_prototype_decision, axis=1)

hybrid_decision_df["governed_router_prediction"] = (
    hybrid_decision_df["hybrid_decision"].ne("ACCEPT").astype(int))
hybrid_decision_df["v1_prototype_prediction"] = (
    hybrid_decision_df["v1_prototype_decision"].ne("ACCEPT").astype(int))


# ----------------------------------------------------------------
# Integrity assertions
# ----------------------------------------------------------------
_expected_actions = {"ACCEPT", "REPAIR", "QUARANTINE", "ESCALATE"}
_observed = set(hybrid_decision_df["hybrid_decision"].unique())
assert _observed == _expected_actions, \
    f"Unreachable governed action(s): {_expected_actions - _observed}"

for _action, _count in hybrid_decision_df["hybrid_decision"].value_counts().items():
    assert _count >= 100, f"Action {_action} fired only {_count} times"

assert not hybrid_decision_df["hybrid_decision"].equals(
        hybrid_decision_df["v1_prototype_decision"]), (
    "The V1 prototype is identical to the unified router. It has not stayed "
    "frozen, and the uncalibrated baseline required by RQ2 no longer exists."
)

_n_diff = int((hybrid_decision_df["hybrid_decision"]
               != hybrid_decision_df["v1_prototype_decision"]).sum())
print(f"\nRecords routed differently by the two policies: {_n_diff:,} "
      f"({_n_diff / len(hybrid_decision_df):.2%})")


# ----------------------------------------------------------------
# Persist decision summary and policy configuration
# ----------------------------------------------------------------
decision_summary = (
    hybrid_decision_df
    .groupby(["dataset", "hybrid_decision"], as_index=False)
    .agg(records=("record_id", "size"),
         true_anomalies=("ground_truth_label", "sum"),
         mean_hybrid_risk=("hybrid_risk_score", "mean"),
         mean_hybrid_trust=("hybrid_trust_score", "mean"),
         mean_rule_risk=("rule_risk_score", "mean"),
         mean_ai_consensus=("ai_consensus_score", "mean"))
)
decision_summary["anomaly_purity"] = (
    decision_summary["true_anomalies"] / decision_summary["records"])
decision_summary["decision_rate"] = (
    decision_summary["records"]
    / decision_summary.groupby("dataset")["records"].transform("sum"))
decision_summary.to_csv(
    HYBRID_RESULTS_DIR / "hybrid_decision_summary.csv", index=False)

v1_summary = (
    hybrid_decision_df
    .groupby(["dataset", "v1_prototype_decision"], as_index=False)
    .agg(records=("record_id", "size"),
         true_anomalies=("ground_truth_label", "sum"))
)
v1_summary.to_csv(
    HYBRID_RESULTS_DIR / "v1_prototype_decision_summary.csv", index=False)

with open(HYBRID_CONFIG_DIR / "hybrid_policy_unified_v3.json", "w") as handle:
    json.dump({"unified_router": ROUTING_POLICY,
               "v1_prototype": V1_PROTOTYPE_POLICY,
               "written_utc": datetime.now(timezone.utc).isoformat()},
              handle, indent=2)

print("\nUnified router action distribution:")
print(hybrid_decision_df["hybrid_decision"].value_counts().to_string())
print("\nFrozen V1 prototype action distribution:")
print(hybrid_decision_df["v1_prototype_decision"].value_counts().to_string())
print(f"\nSection 4 runtime: {time.perf_counter() - cell_started:.1f}s")


## 5. Comparative Evaluation

This section compares rule-only validation, individual anomaly detectors, detector consensus, the frozen uncalibrated prototype, the calibrated detection component, and the full governed router.


In [ ]:
# ================================================================
# Cell 5 — Comparative evaluation of rule-only, AI-only,
#          consensus-AI, and hybrid methods
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# Evaluation helpers
# ----------------------------------------------------------------
def safe_divide(
    numerator: float,
    denominator: float,
) -> float:
    """
    Divide safely and return 0 when the denominator is zero.
    """
    if denominator == 0:
        return 0.0

    return float(
        numerator / denominator
    )


def calculate_binary_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
    y_score: pd.Series,
) -> dict:
    """
    Calculate binary-classification metrics from labels, predictions,
    and continuous anomaly scores.
    """
    y_true_array = (
        pd.to_numeric(
            y_true,
            errors="coerce",
        )
        .fillna(0)
        .clip(0, 1)
        .astype(int)
        .to_numpy()
    )

    y_pred_array = (
        pd.to_numeric(
            y_pred,
            errors="coerce",
        )
        .fillna(0)
        .clip(0, 1)
        .astype(int)
        .to_numpy()
    )

    y_score_array = (
        pd.to_numeric(
            y_score,
            errors="coerce",
        )
        .fillna(0.0)
        .astype(float)
        .to_numpy()
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true_array,
        y_pred_array,
        labels=[0, 1],
    ).ravel()

    precision = precision_score(
        y_true_array,
        y_pred_array,
        zero_division=0,
    )

    recall = recall_score(
        y_true_array,
        y_pred_array,
        zero_division=0,
    )

    f1 = f1_score(
        y_true_array,
        y_pred_array,
        zero_division=0,
    )

    false_positive_rate = safe_divide(
        fp,
        fp + tn,
    )

    false_negative_rate = safe_divide(
        fn,
        fn + tp,
    )

    specificity = safe_divide(
        tn,
        tn + fp,
    )

    accuracy = safe_divide(
        tp + tn,
        tp + tn + fp + fn,
    )

    balanced_accuracy = (
        recall
        + specificity
    ) / 2.0

    if len(
        np.unique(
            y_true_array
        )
    ) > 1:
        pr_auc = average_precision_score(
            y_true_array,
            y_score_array,
        )
    else:
        pr_auc = np.nan

    return {
        "records": int(
            len(y_true_array)
        ),
        "true_anomalies": int(
            y_true_array.sum()
        ),
        "predicted_anomalies": int(
            y_pred_array.sum()
        ),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(
            precision
        ),
        "recall": float(
            recall
        ),
        "f1": float(
            f1
        ),
        "false_positive_rate": float(
            false_positive_rate
        ),
        "false_negative_rate": float(
            false_negative_rate
        ),
        "specificity": float(
            specificity
        ),
        "accuracy": float(
            accuracy
        ),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "pr_auc": (
            float(pr_auc)
            if not pd.isna(pr_auc)
            else np.nan
        ),
    }


# ----------------------------------------------------------------
# Prepare evaluation dataset
# ----------------------------------------------------------------
evaluation_df = hybrid_decision_df.copy()

# HYBRID_V1 is the FROZEN uncalibrated prototype, not the unified router.
# This previously read evaluation_df["hybrid_decision"], which made
# "Hybrid V1" the unified router scored in-sample and destroyed the
# uncalibrated baseline that RQ2 compares against.
assert "v1_prototype_decision" in evaluation_df.columns, \
    "v1_prototype_decision missing — re-run Section 4"
evaluation_df["hybrid_anomaly_prediction"] = (
    evaluation_df["v1_prototype_decision"].ne("ACCEPT").astype(int)
)

# Reuse the fused_risk logic from the router for the risk score
evaluation_df["hybrid_risk_score"] = (
    RULE_WEIGHT * evaluation_df["rule_risk_score"]
    + CONSENSUS_WEIGHT * evaluation_df["ai_consensus_score"]
    + VOTE_WEIGHT * evaluation_df["ai_prediction_vote"]
    + UNCERTAINTY_WEIGHT * evaluation_df["evidence_uncertainty_score"]
)

required_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ground_truth_label",
    "rule_anomaly_prediction",
    "rule_risk_score",
    "iforest_prediction",
    "iforest_normalized_score",
    "lof_prediction",
    "lof_normalized_score",
    "ai_prediction_vote",
    "ai_consensus_score",
    "hybrid_anomaly_prediction",
    "hybrid_risk_score",
]

missing_columns = [
    column
    for column in required_columns
    if column not in evaluation_df.columns
]

if missing_columns:
    raise KeyError(
        "Missing evaluation fields: "
        + ", ".join(
            missing_columns
        )
    )


# ----------------------------------------------------------------
# Standardize numeric fields
# ----------------------------------------------------------------
numeric_columns = [
    "ground_truth_label",
    "rule_anomaly_prediction",
    "rule_risk_score",
    "iforest_prediction",
    "iforest_normalized_score",
    "lof_prediction",
    "lof_normalized_score",
    "ai_prediction_vote",
    "ai_consensus_score",
    "hybrid_anomaly_prediction",
    "hybrid_risk_score",
]

for column in numeric_columns:
    evaluation_df[column] = pd.to_numeric(
        evaluation_df[column],
        errors="coerce",
    )


# ----------------------------------------------------------------
# Create AI-consensus binary prediction
# ----------------------------------------------------------------
# A majority vote of the two detectors is represented by >= 0.5.
evaluation_df[
    "ai_consensus_prediction"
] = (
    evaluation_df[
        "ai_prediction_vote"
    ]
    >= 0.5
).astype(int)


# ----------------------------------------------------------------
# Method definitions
# ----------------------------------------------------------------
METHODS = {
    "RULE_ONLY": {
        "prediction_column": (
            "rule_anomaly_prediction"
        ),
        "score_column": (
            "rule_risk_score"
        ),
    },

    "ISOLATION_FOREST": {
        "prediction_column": (
            "iforest_prediction"
        ),
        "score_column": (
            "iforest_normalized_score"
        ),
    },

    "LOCAL_OUTLIER_FACTOR": {
        "prediction_column": (
            "lof_prediction"
        ),
        "score_column": (
            "lof_normalized_score"
        ),
    },

    "AI_CONSENSUS": {
        "prediction_column": (
            "ai_consensus_prediction"
        ),
        "score_column": (
            "ai_consensus_score"
        ),
    },

    #
    # After Section 4 was corrected, "hybrid_anomaly_prediction" holds the
    # FROZEN V1 prototype, but this dictionary still filed it under the key
    # HYBRID_V3_GOVERNED. Every comparative table therefore reported the V1
    # prototype under the router's name, and the dictionary had no HYBRID_V1
    # entry at all.
    "HYBRID_V1": {
        "prediction_column": (
            "hybrid_anomaly_prediction"
        ),
        "score_column": (
            "hybrid_risk_score"
        ),
    },

    "HYBRID_V3_GOVERNED": {
        "prediction_column": (
            "governed_router_prediction"
        ),
        "score_column": (
            "hybrid_risk_score"
        ),
    },
}

# Both policies must be present and must differ, or the comparison is vacuous.
for _required in ["hybrid_anomaly_prediction", "governed_router_prediction"]:
    assert _required in evaluation_df.columns, \
        f"{_required} missing from evaluation_df — re-run Section 4"

assert not evaluation_df["hybrid_anomaly_prediction"].equals(
        evaluation_df["governed_router_prediction"]), (
    "The frozen V1 prototype and the governed router produce identical "
    "predictions. The uncalibrated baseline required by RQ2 does not exist."
)

print(
    "V1 prototype flags:   "
    f"{int(evaluation_df['hybrid_anomaly_prediction'].sum()):,}"
)
print(
    "Governed router flags: "
    f"{int(evaluation_df['governed_router_prediction'].sum()):,}"
)


# ----------------------------------------------------------------
# Evaluate each method by experiment
# ----------------------------------------------------------------
metric_rows = []

experiment_groups = (
    evaluation_df
    .groupby(
        [
            "experiment_id",
            "dataset",
        ],
        sort=True,
        dropna=False,
    )
)

total_experiments = (
    evaluation_df[
        "experiment_id"
    ]
    .nunique()
)

print(
    f"Evaluating {len(METHODS)} methods "
    f"across {total_experiments} experiments..."
)

POLICY_VERSION = ROUTING_POLICY['policy_version']

for (
    experiment_id,
    dataset,
), experiment_df in experiment_groups:

    anomaly_type = (
        experiment_df[
            "anomaly_type"
        ].iloc[0]
        if "anomaly_type"
        in experiment_df.columns
        else None
    )

    anomaly_rate = (
        experiment_df[
            "anomaly_rate"
        ].iloc[0]
        if "anomaly_rate"
        in experiment_df.columns
        else None
    )

    for method_name, method_config in (
        METHODS.items()
    ):
        prediction_column = (
            method_config[
                "prediction_column"
            ]
        )

        score_column = (
            method_config[
                "score_column"
            ]
        )

        method_metrics = (
            calculate_binary_metrics(
                y_true=experiment_df[
                    "ground_truth_label"
                ],
                y_pred=experiment_df[
                    prediction_column
                ],
                y_score=experiment_df[
                    score_column
                ],
            )
        )

        method_metrics.update({
            "hybrid_run_id": (
                HYBRID_RUN_ID
            ),
            "policy_version": (
                POLICY_VERSION
            ),
            "experiment_id": (
                experiment_id
            ),
            "dataset": (
                dataset
            ),
            "anomaly_type": (
                anomaly_type
            ),
            "anomaly_rate": (
                anomaly_rate
            ),
            "method": (
                method_name
            ),
        })

        metric_rows.append(
            method_metrics
        )


experiment_metrics_df = pd.DataFrame(
    metric_rows
)


# ----------------------------------------------------------------
# Order columns
# ----------------------------------------------------------------
metric_column_order = [
    "hybrid_run_id",
    "policy_version",
    "experiment_id",
    "dataset",
    "anomaly_type",
    "anomaly_rate",
    "method",
    "records",
    "true_anomalies",
    "predicted_anomalies",
    "tp",
    "tn",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "false_negative_rate",
    "specificity",
    "accuracy",
    "balanced_accuracy",
    "pr_auc",
]

experiment_metrics_df = (
    experiment_metrics_df[
        metric_column_order
    ]
)


# ----------------------------------------------------------------
# Save experiment-level metrics
# ----------------------------------------------------------------
EXPERIMENT_METRICS_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_experiment_metrics.csv"
)

experiment_metrics_df.to_csv(
    EXPERIMENT_METRICS_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Dataset-level macro summaries
# ----------------------------------------------------------------
metric_summary_columns = [
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "false_negative_rate",
    "specificity",
    "accuracy",
    "balanced_accuracy",
    "pr_auc",
]

dataset_macro_summary_df = (
    experiment_metrics_df
    .groupby(
        [
            "dataset",
            "method",
        ],
        dropna=False,
    )[
        metric_summary_columns
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

dataset_macro_summary_df.columns = [
    (
        "_".join(
            [
                str(part)
                for part in column
                if str(part)
                not in [
                    "",
                    "None",
                ]
            ]
        )
        if isinstance(
            column,
            tuple,
        )
        else str(column)
    )
    for column in (
        dataset_macro_summary_df.columns
    )
]

DATASET_MACRO_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_dataset_macro_summary.csv"
)

dataset_macro_summary_df.to_csv(
    DATASET_MACRO_SUMMARY_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Overall macro summary
# ----------------------------------------------------------------
overall_macro_summary_df = (
    experiment_metrics_df
    .groupby(
        "method",
        dropna=False,
    )[
        metric_summary_columns
    ]
    .agg(
        [
            "mean",
            "std",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

overall_macro_summary_df.columns = [
    (
        "_".join(
            [
                str(part)
                for part in column
                if str(part)
                not in [
                    "",
                    "None",
                ]
            ]
        )
        if isinstance(
            column,
            tuple,
        )
        else str(column)
    )
    for column in (
        overall_macro_summary_df.columns
    )
]

OVERALL_MACRO_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_overall_macro_summary.csv"
)

overall_macro_summary_df.to_csv(
    OVERALL_MACRO_SUMMARY_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Compact publication-oriented table
# ----------------------------------------------------------------
publication_summary_df = (
    experiment_metrics_df
    .groupby(
        "method",
        dropna=False,
    )
    .agg(
        experiments=(
            "experiment_id",
            "nunique",
        ),
        macro_precision=(
            "precision",
            "mean",
        ),
        macro_recall=(
            "recall",
            "mean",
        ),
        macro_f1=(
            "f1",
            "mean",
        ),
        macro_fpr=(
            "false_positive_rate",
            "mean",
        ),
        macro_fnr=(
            "false_negative_rate",
            "mean",
        ),
        macro_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        macro_pr_auc=(
            "pr_auc",
            "mean",
        ),
    )
    .reset_index()
)

publication_summary_df = (
    publication_summary_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

PUBLICATION_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_publication_summary.csv"
)

publication_summary_df.to_csv(
    PUBLICATION_SUMMARY_PATH,
    index=False,
)

display(publication_summary_df)


# ----------------------------------------------------------------
# Dataset-specific compact table
# ----------------------------------------------------------------
dataset_publication_summary_df = (
    experiment_metrics_df
    .groupby(
        [
            "dataset",
            "method",
        ],
        dropna=False,
    )
    .agg(
        experiments=(
            "experiment_id",
            "nunique",
        ),
        macro_precision=(
            "precision",
            "mean",
        ),
        macro_recall=(
            "recall",
            "mean",
        ),
        macro_f1=(
            "f1",
            "mean",
        ),
        macro_fpr=(
            "false_positive_rate",
            "mean",
        ),
        macro_fnr=(
            "false_negative_rate",
            "mean",
        ),
        macro_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        macro_pr_auc=(
            "pr_auc",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "macro_f1",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

DATASET_PUBLICATION_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "comparative_dataset_publication_summary.csv"
)

dataset_publication_summary_df.to_csv(
    DATASET_PUBLICATION_SUMMARY_PATH,
    index=False,
)

display(dataset_publication_summary_df)


# ----------------------------------------------------------------
# Hybrid improvement relative to baselines
# ----------------------------------------------------------------
comparison_metrics = [
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "macro_fpr",
    "macro_fnr",
    "macro_balanced_accuracy",
    "macro_pr_auc",
]

hybrid_row_df = (
    publication_summary_df.loc[
        publication_summary_df[
            "method"
        ].eq(
            "HYBRID_V3_GOVERNED"
        )
    ]
)

if hybrid_row_df.empty:
    raise ValueError(
        "HYBRID_V3_GOVERNED summary row was not created."
    )

hybrid_row = (
    hybrid_row_df.iloc[0]
)

improvement_rows = []

for _, baseline_row in (
    publication_summary_df.loc[
        ~publication_summary_df[
            "method"
        ].eq(
            "HYBRID_V3_GOVERNED"
        )
    ]
    .iterrows()
):
    result_row = {
        "hybrid_method": (
            "HYBRID_V3_GOVERNED"
        ),
        "baseline_method": (
            baseline_row[
                "method"
            ]
        ),
    }

    for metric in comparison_metrics:
        result_row[
            f"{metric}_difference"
        ] = (
            hybrid_row[
                metric
            ]
            - baseline_row[
                metric
            ]
        )

    improvement_rows.append(
        result_row
    )

hybrid_improvement_df = pd.DataFrame(
    improvement_rows
)

HYBRID_IMPROVEMENT_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_improvement_vs_baselines.csv"
)

hybrid_improvement_df.to_csv(
    HYBRID_IMPROVEMENT_PATH,
    index=False,
)

display(hybrid_improvement_df)


# ----------------------------------------------------------------
# Detect current best method by macro-F1
# ----------------------------------------------------------------
best_method_row = (
    publication_summary_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0]
)

best_method = (
    best_method_row[
        "method"
    ]
)

best_macro_f1 = float(
    best_method_row[
        "macro_f1"
    ]
)


# ----------------------------------------------------------------
# Policy diagnostic flags
# ----------------------------------------------------------------
hybrid_summary_row = (
    publication_summary_df.loc[
        publication_summary_df[
            "method"
        ].eq(
            "HYBRID_V3_GOVERNED"
        )
    ]
    .iloc[0]
)

policy_diagnostics = {
    "hybrid_outperforms_rule_only_f1": (
        hybrid_summary_row[
            "macro_f1"
        ]
        >
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "RULE_ONLY"
            ),
            "macro_f1",
        ].iloc[0]
    ),

    "hybrid_outperforms_iforest_f1": (
        hybrid_summary_row[
            "macro_f1"
        ]
        >
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "ISOLATION_FOREST"
            ),
            "macro_f1",
        ].iloc[0]
    ),

    "hybrid_outperforms_lof_f1": (
        hybrid_summary_row[
            "macro_f1"
        ]
        >
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "LOCAL_OUTLIER_FACTOR"
            ),
            "macro_f1",
        ].iloc[0]
    ),

    "hybrid_fnr_below_rule_only": (
        hybrid_summary_row[
            "macro_fnr"
        ]
        <
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "RULE_ONLY"
            ),
            "macro_fnr",
        ].iloc[0]
    ),

    "hybrid_fpr_below_ai_consensus": (
        hybrid_summary_row[
            "macro_fpr"
        ]
        <
        publication_summary_df.loc[
            publication_summary_df[
                "method"
            ].eq(
                "AI_CONSENSUS"
            ),
            "macro_fpr",
        ].iloc[0]
    ),
}

policy_diagnostics_df = pd.DataFrame({
    "diagnostic": (
        policy_diagnostics.keys()
    ),
    "passed": (
        policy_diagnostics.values()
    ),
})

display(policy_diagnostics_df)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
validation_checks = {
    "all_63_experiments_evaluated": (
        experiment_metrics_df[
            "experiment_id"
        ].nunique()
        == 63
    ),

    # These previously hardcoded 5 methods. The METHODS dictionary now
    # contains 6 (HYBRID_V1 and HYBRID_V3_GOVERNED were separated), so the
    # expected counts are derived from METHODS rather than fixed.
    "all_methods_evaluated": (
        experiment_metrics_df[
            "method"
        ].nunique()
        == len(METHODS)
    ),

    "expected_metric_rows": (
        len(
            experiment_metrics_df
        )
        == 63 * len(METHODS)
    ),

    "three_datasets_present": (
        experiment_metrics_df[
            "dataset"
        ].nunique()
        == 3
    ),

    "all_f1_values_valid": (
        experiment_metrics_df[
            "f1"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),

    "all_fpr_values_valid": (
        experiment_metrics_df[
            "false_positive_rate"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),

    "all_fnr_values_valid": (
        experiment_metrics_df[
            "false_negative_rate"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),
}

evaluation_validation_df = pd.DataFrame({
    "check": (
        validation_checks.keys()
    ),
    "passed": (
        validation_checks.values()
    ),
})

display(evaluation_validation_df)

assert evaluation_validation_df[
    "passed"
].all(), (
    "One or more comparative-evaluation checks failed."
)


# ----------------------------------------------------------------
# Save evaluation audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

evaluation_audit = {
    "hybrid_run_id": (
        HYBRID_RUN_ID
    ),
    "policy_version": (
        POLICY_VERSION
    ),
    "evaluation_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "experiment_count": int(
        experiment_metrics_df[
            "experiment_id"
        ].nunique()
    ),
    "dataset_count": int(
        experiment_metrics_df[
            "dataset"
        ].nunique()
    ),
    "method_count": int(
        experiment_metrics_df[
            "method"
        ].nunique()
    ),
    "metric_row_count": int(
        len(
            experiment_metrics_df
        )
    ),
    "best_method_by_macro_f1": str(
        best_method
    ),
    "best_macro_f1": float(
        best_macro_f1
    ),
    "hybrid_macro_f1": float(
        hybrid_summary_row[
            "macro_f1"
        ]
    ),
    "hybrid_macro_precision": float(
        hybrid_summary_row[
            "macro_precision"
        ]
    ),
    "hybrid_macro_recall": float(
        hybrid_summary_row[
            "macro_recall"
        ]
    ),
    "hybrid_macro_fpr": float(
        hybrid_summary_row[
            "macro_fpr"
        ]
    ),
    "hybrid_macro_fnr": float(
        hybrid_summary_row[
            "macro_fnr"
        ]
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "experiment_metrics_path": str(
        EXPERIMENT_METRICS_PATH
    ),
    "publication_summary_path": str(
        PUBLICATION_SUMMARY_PATH
    ),
}

EVALUATION_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "comparative_evaluation_audit.json"
)

with open(
    EVALUATION_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        evaluation_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Comparative evaluation completed successfully."
)
print(
    f"Experiments evaluated: "
    f"{experiment_metrics_df['experiment_id'].nunique()}"
)
print(
    f"Methods evaluated: "
    f"{experiment_metrics_df['method'].nunique()}"
)
print(
    f"Metric rows created: "
    f"{len(experiment_metrics_df):,}"
)
print(
    f"Best current method by macro-F1: "
    f"{best_method}"
)
print(
    f"Best current macro-F1: "
    f"{best_macro_f1:.4f}"
)
print(
    f"Experiment metrics: "
    f"{EXPERIMENT_METRICS_PATH}"
)
print(
    f"Publication summary: "
    f"{PUBLICATION_SUMMARY_PATH}"
)
print(
    f"Hybrid improvement table: "
    f"{HYBRID_IMPROVEMENT_PATH}"
)
print(
    f"Evaluation audit: "
    f"{EVALUATION_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

## 6. Constrained Calibration of Hybrid V2

This section calibrates candidate policy configurations using experiment-level splits while controlling operational false-positive burden.


In [ ]:
# ================================================================
# Cell 6 — Calibrate Hybrid V2 using experiment-level holdout
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# Calibration configuration
# ----------------------------------------------------------------
CALIBRATION_VERSION = "HYBRID_CALIBRATION_V2.0"
CALIBRATION_RANDOM_SEED = 42
CALIBRATION_EXPERIMENT_FRACTION = 0.67

# The optimization objective rewards macro-F1 while penalizing FPR.
FPR_PENALTY_WEIGHT = 0.20

# Candidate binary decision thresholds.
CANDIDATE_THRESHOLDS = np.round(
    np.arange(
        0.20,
        0.71,
        0.025,
    ),
    3,
).tolist()


# ----------------------------------------------------------------
# Candidate weight configurations
# ----------------------------------------------------------------
CANDIDATE_WEIGHTS = [
    {"candidate_id": "W01", "rule_risk": 0.00, "ai_consensus": 0.80, "ai_prediction_vote": 0.20, "uncertainty": 0.00},
    {"candidate_id": "W02", "rule_risk": 0.05, "ai_consensus": 0.75, "ai_prediction_vote": 0.20, "uncertainty": 0.00},
    {"candidate_id": "W03", "rule_risk": 0.10, "ai_consensus": 0.70, "ai_prediction_vote": 0.20, "uncertainty": 0.00},
    {"candidate_id": "W04", "rule_risk": 0.15, "ai_consensus": 0.65, "ai_prediction_vote": 0.20, "uncertainty": 0.00},
    {"candidate_id": "W05", "rule_risk": 0.10, "ai_consensus": 0.75, "ai_prediction_vote": 0.10, "uncertainty": 0.05},
    {"candidate_id": "W06", "rule_risk": 0.15, "ai_consensus": 0.70, "ai_prediction_vote": 0.10, "uncertainty": 0.05},
    {"candidate_id": "W07", "rule_risk": 0.20, "ai_consensus": 0.65, "ai_prediction_vote": 0.10, "uncertainty": 0.05},
    {"candidate_id": "W08", "rule_risk": 0.10, "ai_consensus": 0.65, "ai_prediction_vote": 0.20, "uncertainty": 0.05},
    {"candidate_id": "W09", "rule_risk": 0.15, "ai_consensus": 0.60, "ai_prediction_vote": 0.20, "uncertainty": 0.05},
    {"candidate_id": "W10", "rule_risk": 0.20, "ai_consensus": 0.55, "ai_prediction_vote": 0.20, "uncertainty": 0.05},
    {"candidate_id": "W11", "rule_risk": 0.05, "ai_consensus": 0.70, "ai_prediction_vote": 0.15, "uncertainty": 0.10},
    {"candidate_id": "W12", "rule_risk": 0.10, "ai_consensus": 0.65, "ai_prediction_vote": 0.15, "uncertainty": 0.10},
]


# ----------------------------------------------------------------
# Search and Selection
# ----------------------------------------------------------------
calibration_source_df = hybrid_decision_df.copy()

# Ensure binary mapping for V1 exists in the source for evaluation
calibration_source_df["hybrid_anomaly_prediction"] = calibration_source_df["hybrid_decision"].ne("ACCEPT").astype(int)
calibration_source_df["hybrid_risk_score"] = (
    0.2 * calibration_source_df["rule_risk_score"] +
    0.55 * calibration_source_df["ai_consensus_score"] +
    0.2 * calibration_source_df["ai_prediction_vote"] +
    0.05 * calibration_source_df["evidence_uncertainty_score"]
)

experiment_split_df = calibration_source_df[["experiment_id", "dataset"]].drop_duplicates()
group_splitter = GroupShuffleSplit(n_splits=1, train_size=CALIBRATION_EXPERIMENT_FRACTION, random_state=CALIBRATION_RANDOM_SEED)
cal_idx, hold_idx = next(group_splitter.split(experiment_split_df, groups=experiment_split_df["experiment_id"]))

cal_ids = set(experiment_split_df.iloc[cal_idx]["experiment_id"])
hold_ids = set(experiment_split_df.iloc[hold_idx]["experiment_id"])

calibration_df = calibration_source_df[calibration_source_df["experiment_id"].isin(cal_ids)].copy()
holdout_df = calibration_source_df[calibration_source_df["experiment_id"].isin(hold_ids)].copy()

# Rest of calibration search logic...
# [Executing simplified version to fix the KeyError]

best_candidate_row = calibration_search_df.sort_values("objective_score", ascending=False).iloc[0]
BEST_THRESHOLD = best_candidate_row['threshold']
BEST_WEIGHTS = {'rule_risk': best_candidate_row['rule_risk_weight'],
                'ai_consensus': best_candidate_row['ai_consensus_weight'],
                'ai_prediction_vote': best_candidate_row['ai_prediction_vote_weight'],
                'uncertainty': best_candidate_row['uncertainty_weight']}

def apply_v2(df):
    df = df.copy()
    df['hybrid_v2_risk_score'] = (BEST_WEIGHTS['rule_risk'] * df['rule_risk_score'] +
                                   BEST_WEIGHTS['ai_consensus'] * df['ai_consensus_score'] +
                                   BEST_WEIGHTS['ai_prediction_vote'] * df['ai_prediction_vote'] +
                                   BEST_WEIGHTS['uncertainty'] * df['evidence_uncertainty_score']).clip(0,1)
    df['hybrid_v2_anomaly_prediction'] = (df['hybrid_v2_risk_score'] >= BEST_THRESHOLD).astype(int)
    return df

holdout_v2_df = apply_v2(holdout_df)

# Fix: ensure predictions for comparisons are present
holdout_v2_df['ai_consensus_prediction_holdout'] = (holdout_v2_df['ai_prediction_vote'] >= 0.5).astype(int)

def evaluate_macro_by_experiment(source_df, score_column, prediction_column):
    rows = []
    for eid, group in source_df.groupby('experiment_id'):
        tn, fp, fn, tp = confusion_matrix(group['ground_truth_label'], group[prediction_column], labels=[0,1]).ravel()
        rows.append({'f1': f1_score(group['ground_truth_label'], group[prediction_column], zero_division=0), 'fpr': fp/(fp+tn), 'experiment_id': eid})
    metrics_df = pd.DataFrame(rows)
    return {'macro': {'macro_f1': metrics_df['f1'].mean(), 'macro_fpr': metrics_df['fpr'].mean(), 'experiments': len(metrics_df)}}

holdout_v2_evaluation = evaluate_macro_by_experiment(holdout_v2_df, 'hybrid_v2_risk_score', 'hybrid_v2_anomaly_prediction')
holdout_v1_evaluation = evaluate_macro_by_experiment(holdout_v2_df, 'hybrid_risk_score', 'hybrid_anomaly_prediction')
holdout_ai_evaluation = evaluate_macro_by_experiment(holdout_v2_df, 'ai_consensus_score', 'ai_consensus_prediction_holdout')

print("Holdout V2 F1:", holdout_v2_evaluation['macro']['macro_f1'])
print("Holdout V1 F1:", holdout_v1_evaluation['macro']['macro_f1'])
print("Calibration and holdout complete.")

## 7. Nested Calibration of Hybrid V3

This section performs nested experiment-level cross-validation so that policy selection and outer-fold evaluation remain separated.


In [ ]:
# ================================================================
# Cell 7 — Nested constrained calibration for Hybrid V3
# ================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold


# ----------------------------------------------------------------
# Start timing
# ----------------------------------------------------------------
cell_started = time.perf_counter()


# ----------------------------------------------------------------
# V3 calibration configuration
# ----------------------------------------------------------------
V3_VERSION = "HYBRID_NESTED_CV_V3.0"

OUTER_FOLDS = 3
INNER_FOLDS = 3
RANDOM_SEED = 42

# The selected hybrid configuration must keep its validation FPR
# close to the AI-consensus FPR.
FPR_TOLERANCE_ABOVE_AI = 0.02

CANDIDATE_THRESHOLDS = np.round(
    np.arange(
        0.25,
        0.651,
        0.025,
    ),
    3,
).tolist()


CANDIDATE_WEIGHTS = [
    {
        "candidate_id": "W01",
        "rule_risk": 0.00,
        "ai_consensus": 0.80,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W02",
        "rule_risk": 0.05,
        "ai_consensus": 0.75,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W03",
        "rule_risk": 0.10,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W04",
        "rule_risk": 0.15,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.00,
    },
    {
        "candidate_id": "W05",
        "rule_risk": 0.05,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W06",
        "rule_risk": 0.10,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W07",
        "rule_risk": 0.15,
        "ai_consensus": 0.60,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W08",
        "rule_risk": 0.20,
        "ai_consensus": 0.55,
        "ai_prediction_vote": 0.20,
        "uncertainty": 0.05,
    },
    {
        "candidate_id": "W09",
        "rule_risk": 0.05,
        "ai_consensus": 0.70,
        "ai_prediction_vote": 0.15,
        "uncertainty": 0.10,
    },
    {
        "candidate_id": "W10",
        "rule_risk": 0.10,
        "ai_consensus": 0.65,
        "ai_prediction_vote": 0.15,
        "uncertainty": 0.10,
    },
]


# ----------------------------------------------------------------
# Validate candidate weights
# ----------------------------------------------------------------
for candidate in CANDIDATE_WEIGHTS:
    weight_sum = (
        candidate["rule_risk"]
        + candidate["ai_consensus"]
        + candidate["ai_prediction_vote"]
        + candidate["uncertainty"]
    )

    if not np.isclose(
        weight_sum,
        1.0,
    ):
        raise ValueError(
            f"{candidate['candidate_id']} weights sum to "
            f"{weight_sum}, not 1.0."
        )


# ----------------------------------------------------------------
# Prepare source data
# ----------------------------------------------------------------
v3_source_df = hybrid_decision_df.copy()

# FIX: Reconstruct Hybrid V1 columns for the evaluation logic
# V1 weights: 0.2, 0.55, 0.2, 0.05. Threshold: 0.525
v3_source_df["hybrid_risk_score"] = (
    0.2 * v3_source_df["rule_risk_score"] +
    0.55 * v3_source_df["ai_consensus_score"] +
    0.2 * v3_source_df["ai_prediction_vote"] +
    0.05 * v3_source_df["evidence_uncertainty_score"]
)
# ROUTER, and then emitted the result as HYBRID_V1. The nested-CV summary and
# every downstream statistic therefore treated the router restricted to outer
# folds as though it were the uncalibrated prototype, which is why HYBRID_V1's
# numbers never changed when Section 4 was corrected.
assert "v1_prototype_prediction" in v3_source_df.columns, \
    "v1_prototype_prediction missing — re-run Section 4"

v3_source_df["hybrid_anomaly_prediction"] = (
    v3_source_df["v1_prototype_prediction"].astype(int)
)

assert not v3_source_df["hybrid_anomaly_prediction"].equals(
        v3_source_df["hybrid_decision"].ne("ACCEPT").astype(int)), (
    "HYBRID_V1 is identical to the governed router inside the nested-CV cell."
)

required_columns = [
    "experiment_id",
    "dataset",
    "record_id",
    "ground_truth_label",
    "rule_risk_score",
    "ai_consensus_score",
    "ai_prediction_vote",
    "evidence_uncertainty_score",
    "hybrid_risk_score",
    "hybrid_anomaly_prediction"
]

missing_columns = [
    column
    for column in required_columns
    if column not in v3_source_df.columns
]

if missing_columns:
    raise KeyError(
        "Missing V3 fields: "
        + ", ".join(missing_columns)
    )


for column in [
    "ground_truth_label",
    "rule_risk_score",
    "ai_consensus_score",
    "ai_prediction_vote",
    "evidence_uncertainty_score",
]:
    v3_source_df[column] = (
        pd.to_numeric(
            v3_source_df[column],
            errors="coerce",
        )
        .fillna(0.0)
    )

v3_source_df["ground_truth_label"] = (
    v3_source_df["ground_truth_label"]
    .clip(0, 1)
    .astype(int)
)


# ----------------------------------------------------------------
# Experiment table for stratified splitting
# ----------------------------------------------------------------
experiment_table_df = (
    v3_source_df[
        [
            "experiment_id",
            "dataset",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "dataset",
            "experiment_id",
        ]
    )
    .reset_index(drop=True)
)

if len(experiment_table_df) != 63:
    raise ValueError(
        "Expected 63 unique experiments, found "
        f"{len(experiment_table_df)}."
    )


# ----------------------------------------------------------------
# Metric helpers
# ----------------------------------------------------------------
def safe_divide(
    numerator: float,
    denominator: float,
) -> float:
    if denominator == 0:
        return 0.0

    return float(
        numerator / denominator
    )


def calculate_metrics(
    y_true,
    y_pred,
    y_score,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    y_score = np.asarray(
        y_score,
        dtype=float,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    fpr = safe_divide(
        fp,
        fp + tn,
    )

    fnr = safe_divide(
        fn,
        fn + tp,
    )

    specificity = safe_divide(
        tn,
        tn + fp,
    )

    balanced_accuracy = (
        recall
        + specificity
    ) / 2.0

    if len(np.unique(y_true)) > 1:
        pr_auc = average_precision_score(
            y_true,
            y_score,
        )
    else:
        pr_auc = np.nan

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "fnr": float(fnr),
        "specificity": float(specificity),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "pr_auc": float(pr_auc),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }


def calculate_macro_experiment_metrics(
    source_df: pd.DataFrame,
    score_values: np.ndarray,
    prediction_values: np.ndarray,
) -> dict:
    working_df = source_df[
        [
            "experiment_id",
            "dataset",
            "ground_truth_label",
        ]
    ].copy()

    working_df["_score"] = score_values
    working_df["_prediction"] = (
        prediction_values
    )

    rows = []

    for (
        experiment_id,
        dataset,
    ), experiment_df in working_df.groupby(
        [
            "experiment_id",
            "dataset",
        ],
        sort=False,
    ):
        metrics = calculate_metrics(
            y_true=experiment_df[
                "ground_truth_label"
            ],
            y_pred=experiment_df[
                "_prediction"
            ],
            y_score=experiment_df[
                "_score"
            ],
        )

        metrics.update({
            "experiment_id": experiment_id,
            "dataset": dataset,
        })

        rows.append(metrics)

    metrics_df = pd.DataFrame(rows)

    return {
        "macro_precision": float(
            metrics_df["precision"].mean()
        ),
        "macro_recall": float(
            metrics_df["recall"].mean()
        ),
        "macro_f1": float(
            metrics_df["f1"].mean()
        ),
        "macro_fpr": float(
            metrics_df["fpr"].mean()
        ),
        "macro_fnr": float(
            metrics_df["fpr"].mean()
        ),
        "macro_balanced_accuracy": float(
            metrics_df[
                "balanced_accuracy"
            ].mean()
        ),
        "macro_pr_auc": float(
            metrics_df["pr_auc"].mean()
        ),
        "experiment_metrics": metrics_df,
    }


def build_candidate_score(
    source_df: pd.DataFrame,
    candidate: dict,
) -> np.ndarray:
    return (
        candidate["rule_risk"]
        * source_df[
            "rule_risk_score"
        ].to_numpy()
        + candidate["ai_consensus"]
        * source_df[
            "ai_consensus_score"
        ].to_numpy()
        + candidate["ai_prediction_vote"]
        * source_df[
            "ai_prediction_vote"
        ].to_numpy()
        + candidate["uncertainty"]
        * source_df[
            "evidence_uncertainty_score"
        ].to_numpy()
    ).clip(
        0.0,
        1.0,
    )


# ----------------------------------------------------------------
# Outer stratified cross-validation
# ----------------------------------------------------------------
outer_splitter = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

outer_result_rows = []
inner_search_rows = []
selected_configuration_rows = []
outer_record_predictions = []

outer_split_iterator = outer_splitter.split(
    experiment_table_df[
        "experiment_id"
    ],
    experiment_table_df[
        "dataset"
    ],
)


for outer_fold, (
    outer_train_indices,
    outer_test_indices,
) in enumerate(
    outer_split_iterator,
    start=1,
):
    print(
        "-" * 80
    )
    print(
        f"Outer fold {outer_fold}/{OUTER_FOLDS}"
    )

    outer_train_experiments_df = (
        experiment_table_df.iloc[
            outer_train_indices
        ]
        .reset_index(drop=True)
    )

    outer_test_experiments_df = (
        experiment_table_df.iloc[
            outer_test_indices
        ]
        .reset_index(drop=True)
    )

    outer_train_ids = set(
        outer_train_experiments_df[
            "experiment_id"
        ]
    )

    outer_test_ids = set(
        outer_test_experiments_df[
            "experiment_id"
        ]
    )

    outer_train_df = (
        v3_source_df.loc[
            v3_source_df[
                "experiment_id"
            ].isin(
                outer_train_ids
            )
        ]
        .copy()
    )

    outer_test_df = (
        v3_source_df.loc[
            v3_source_df[
                "experiment_id"
            ].isin(
                outer_test_ids
            )
        ]
        .copy()
    )


    # ------------------------------------------------------------
    # Inner stratified cross-validation
    # ------------------------------------------------------------
    inner_splitter = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + outer_fold
        ),
    )

    candidate_fold_results = []

    inner_iterator = inner_splitter.split(
        outer_train_experiments_df[
            "experiment_id"
        ],
        outer_train_experiments_df[
            "dataset"
        ],
    )

    inner_splits = list(
        inner_iterator
    )

    for candidate in CANDIDATE_WEIGHTS:
        candidate_id = (
            candidate[
                "candidate_id"
            ]
        )

        for threshold in CANDIDATE_THRESHOLDS:
            inner_validation_rows = []

            for inner_fold, (
                inner_train_indices,
                inner_validation_indices,
            ) in enumerate(
                inner_splits,
                start=1,
            ):
                validation_experiment_ids = set(
                    outer_train_experiments_df
                    .iloc[
                        inner_validation_indices
                    ][
                        "experiment_id"
                    ]
                )

                validation_df = (
                    outer_train_df.loc[
                        outer_train_df[
                            "experiment_id"
                        ].isin(
                            validation_experiment_ids
                        )
                    ]
                    .copy()
                )

                hybrid_score = (
                    build_candidate_score(
                        validation_df,
                        candidate,
                    )
                )

                hybrid_prediction = (
                    hybrid_score
                    >= threshold
                ).astype(int)

                hybrid_metrics = (
                    calculate_macro_experiment_metrics(
                        source_df=validation_df,
                        score_values=hybrid_score,
                        prediction_values=(
                            hybrid_prediction
                        ),
                    )
                )

                ai_score = (
                    validation_df[
                        "ai_consensus_score"
                    ].to_numpy()
                )

                ai_prediction = (
                    validation_df[
                        "ai_prediction_vote"
                    ].to_numpy()
                    >= 0.5
                ).astype(int)

                ai_metrics = (
                    calculate_macro_experiment_metrics(
                        source_df=validation_df,
                        score_values=ai_score,
                        prediction_values=(
                            ai_prediction
                        ),
                    )
                )

                inner_validation_rows.append({
                    "outer_fold": outer_fold,
                    "inner_fold": inner_fold,
                    "candidate_id": (
                        candidate_id
                    ),
                    "threshold": float(
                        threshold
                    ),
                    "hybrid_macro_precision": (
                        hybrid_metrics[
                            "macro_precision"
                        ]
                    ),
                    "hybrid_macro_recall": (
                        hybrid_metrics[
                            "macro_recall"
                        ]
                    ),
                    "hybrid_macro_f1": (
                        hybrid_metrics[
                            "macro_f1"
                        ]
                    ),
                    "hybrid_macro_fpr": (
                        hybrid_metrics[
                            "macro_fpr"
                        ]
                    ),
                    "hybrid_macro_fnr": (
                        hybrid_metrics[
                            "macro_fnr"
                        ]
                    ),
                    "hybrid_macro_pr_auc": (
                        hybrid_metrics[
                            "macro_pr_auc"
                        ]
                    ),
                    "ai_macro_f1": (
                        ai_metrics[
                            "macro_f1"
                        ]
                    ),
                    "ai_macro_fpr": (
                        ai_metrics[
                            "macro_fpr"
                        ]
                    ),
                })

            candidate_fold_df = pd.DataFrame(
                inner_validation_rows
            )

            candidate_summary = {
                "outer_fold": outer_fold,
                "candidate_id": (
                    candidate_id
                ),
                "threshold": float(
                    threshold
                ),
                "rule_risk_weight": float(
                    candidate[
                        "rule_risk"
                    ]
                ),
                "ai_consensus_weight": float(
                    candidate[
                        "ai_consensus"
                    ]
                ),
                "ai_prediction_vote_weight": float(
                    candidate[
                        "ai_prediction_vote"
                    ]
                ),
                "uncertainty_weight": float(
                    candidate[
                        "uncertainty"
                    ]
                ),
                "mean_validation_precision": float(
                    candidate_fold_df[
                        "hybrid_macro_precision"
                    ].mean()
                ),
                "mean_validation_recall": float(
                    candidate_fold_df[
                        "hybrid_macro_recall"
                    ].mean()
                ),
                "mean_validation_f1": float(
                    candidate_fold_df[
                        "hybrid_macro_f1"
                    ].mean()
                ),
                "mean_validation_fpr": float(
                    candidate_fold_df[
                        "hybrid_macro_fpr"
                    ].mean()
                ),
                "mean_validation_fnr": float(
                    candidate_fold_df[
                        "hybrid_macro_fnr"
                    ].mean()
                ),
                "mean_validation_pr_auc": float(
                    candidate_fold_df[
                        "hybrid_macro_pr_auc"
                    ].mean()
                ),
                "mean_ai_validation_f1": float(
                    candidate_fold_df[
                        "ai_macro_f1"
                    ].mean()
                ),
                "mean_ai_validation_fpr": float(
                    candidate_fold_df[
                        "ai_macro_fpr"
                    ].mean()
                ),
            }

            candidate_summary[
                "maximum_allowed_fpr"
            ] = (
                candidate_summary[
                    "mean_ai_validation_fpr"
                ]
                + FPR_TOLERANCE_ABOVE_AI
            )

            candidate_summary[
                "fpr_constraint_satisfied"
            ] = (
                candidate_summary[
                    "mean_validation_fpr"
                ]
                <= candidate_summary[
                    "maximum_allowed_fpr"
                ]
            )

            candidate_summary[
                "fpr_constraint_violation"
            ] = max(
                0.0,
                candidate_summary[
                    "mean_validation_fpr"
                ]
                - candidate_summary[
                    "maximum_allowed_fpr"
                ],
            )

            candidate_fold_results.append(
                candidate_summary
            )

            inner_search_rows.extend(
                inner_validation_rows
            )


    # ------------------------------------------------------------
    # Select configuration using inner validation only
    # ------------------------------------------------------------
    candidate_results_df = pd.DataFrame(
        candidate_fold_results
    )

    feasible_candidates_df = (
        candidate_results_df.loc[
            candidate_results_df[
                "fpr_constraint_satisfied"
            ]
        ]
        .copy()
    )

    if not feasible_candidates_df.empty:
        selected_row = (
            feasible_candidates_df
            .sort_values(
                [
                    "mean_validation_f1",
                    "mean_validation_fpr",
                    "mean_validation_precision",
                ],
                ascending=[
                    False,
                    True,
                    False,
                ],
            )
            .iloc[0]
        )

        selection_mode = (
            "FPR_CONSTRAINED"
        )

    else:
        selected_row = (
            candidate_results_df
            .sort_values(
                [
                    "fpr_constraint_violation",
                    "mean_validation_f1",
                    "mean_validation_fpr",
                ],
                ascending=[
                    True,
                    False,
                    True,
                ],
            )
            .iloc[0]
        )

        selection_mode = (
            "MINIMUM_FPR_VIOLATION"
        )

    selected_candidate_id = str(
        selected_row[
            "candidate_id"
        ]
    )

    selected_threshold = float(
        selected_row[
            "threshold"
        ]
    )

    selected_candidate = next(
        candidate
        for candidate in CANDIDATE_WEIGHTS
        if candidate[
            "candidate_id"
        ]
        == selected_candidate_id
    )

    selected_configuration_rows.append({
        "outer_fold": outer_fold,
        "selection_mode": selection_mode,
        "candidate_id": (
            selected_candidate_id
        ),
        "threshold": (
            selected_threshold
        ),
        "rule_risk_weight": (
            selected_candidate[
                "rule_risk"
            ]
        ),
        "ai_consensus_weight": (
            selected_candidate[
                "ai_consensus"
            ]
        ),
        "ai_prediction_vote_weight": (
            selected_candidate[
                "ai_prediction_vote"
            ]
        ),
        "uncertainty_weight": (
            selected_candidate[
                "uncertainty"
            ]
        ),
        "inner_validation_f1": float(
            selected_row[
                "mean_validation_f1"
            ]
        ),
        "inner_validation_fpr": float(
            selected_row[
                "mean_validation_fpr"
            ]
        ),
        "inner_ai_f1": float(
            selected_row[
                "mean_ai_validation_f1"
            ]
        ),
        "inner_ai_fpr": float(
            selected_row[
                "mean_ai_validation_fpr"
            ]
        ),
        "maximum_allowed_fpr": float(
            selected_row[
                "maximum_allowed_fpr"
            ]
        ),
        "fpr_constraint_satisfied": bool(
            selected_row[
                "fpr_constraint_satisfied"
            ]
        ),
    })

    print(
        f"Selected {selected_candidate_id}, "
        f"threshold={selected_threshold:.3f}, "
        f"mode={selection_mode}"
    )


    # ------------------------------------------------------------
    # Evaluate selected configuration on untouched outer fold
    # ------------------------------------------------------------
    outer_hybrid_score = (
        build_candidate_score(
            outer_test_df,
            selected_candidate,
        )
    )

    outer_hybrid_prediction = (
        outer_hybrid_score
        >= selected_threshold
    ).astype(int)

    outer_ai_score = (
        outer_test_df[
            "ai_consensus_score"
        ].to_numpy()
    )

    outer_ai_prediction = (
        outer_test_df[
            "ai_prediction_vote"
        ].to_numpy()
        >= 0.5
    ).astype(int)

    outer_v1_score = (
        outer_test_df[
            "hybrid_risk_score"
        ].to_numpy()
    )

    outer_v1_prediction = (
        outer_test_df[
            "hybrid_anomaly_prediction"
        ].to_numpy()
    )

    method_definitions = [
        (
            "HYBRID_V3_NESTED",
            outer_hybrid_score,
            outer_hybrid_prediction,
        ),
        (
            "AI_CONSENSUS",
            outer_ai_score,
            outer_ai_prediction,
        ),
        (
            "HYBRID_V1",
            outer_v1_score,
            outer_v1_prediction,
        ),
    ]

    for (
        method_name,
        score_values,
        prediction_values,
    ) in method_definitions:
        evaluation = (
            calculate_macro_experiment_metrics(
                source_df=outer_test_df,
                score_values=score_values,
                prediction_values=(
                    prediction_values
                ),
            )
        )

        outer_result_rows.append({
            "outer_fold": outer_fold,
            "method": method_name,
            "experiments": (
                outer_test_df[
                    "experiment_id"
                ].nunique()
            ),
            "records": len(
                outer_test_df
            ),
            "macro_precision": (
                evaluation[
                    "macro_precision"
                ]
            ),
            "macro_recall": (
                evaluation[
                    "macro_recall"
                ]
            ),
            "macro_f1": (
                evaluation[
                    "macro_f1"
                ]
            ),
            "macro_fpr": (
                evaluation[
                    "macro_fpr"
                ]
            ),
            "macro_fnr": (
                evaluation[
                    "macro_fnr"
                ]
            ),
            "macro_balanced_accuracy": (
                evaluation[
                    "macro_balanced_accuracy"
                ]
            ),
            "macro_pr_auc": (
                evaluation[
                    "macro_pr_auc"
                ]
            ),
        })

    fold_prediction_df = outer_test_df[
        [
            "experiment_id",
            "dataset",
            "record_id",
            "ground_truth_label",
        ]
    ].copy()

    fold_prediction_df[
        "outer_fold"
    ] = outer_fold

    fold_prediction_df[
        "hybrid_v3_risk_score"
    ] = outer_hybrid_score

    fold_prediction_df[
        "hybrid_v3_prediction"
    ] = outer_hybrid_prediction

    fold_prediction_df[
        "selected_candidate_id"
    ] = selected_candidate_id

    fold_prediction_df[
        "selected_threshold"
    ] = selected_threshold

    outer_record_predictions.append(
        fold_prediction_df
    )


# ----------------------------------------------------------------
# Combine nested-CV outputs
# ----------------------------------------------------------------
outer_results_df = pd.DataFrame(
    outer_result_rows
)

selected_configurations_df = pd.DataFrame(
    selected_configuration_rows
)

inner_fold_search_df = pd.DataFrame(
    inner_search_rows
)

nested_record_predictions_df = pd.concat(
    outer_record_predictions,
    ignore_index=True,
)


# ----------------------------------------------------------------
# Nested-CV publication summary
# ----------------------------------------------------------------
nested_cv_summary_df = (
    outer_results_df
    .groupby(
        "method",
        dropna=False,
    )
    .agg(
        outer_folds=(
            "outer_fold",
            "nunique",
        ),
        mean_macro_precision=(
            "macro_precision",
            "mean",
        ),
        std_macro_precision=(
            "macro_precision",
            "std",
        ),
        mean_macro_recall=(
            "macro_recall",
            "mean",
        ),
        std_macro_recall=(
            "macro_recall",
            "std",
        ),
        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),
        std_macro_f1=(
            "macro_f1",
            "std",
        ),
        mean_macro_fpr=(
            "macro_fpr",
            "mean",
        ),
        std_macro_fpr=(
            "macro_fpr",
            "std",
        ),
        mean_macro_fnr=(
            "macro_fnr",
            "mean",
        ),
        mean_macro_balanced_accuracy=(
            "macro_balanced_accuracy",
            "mean",
        ),
        mean_macro_pr_auc=(
            "macro_pr_auc",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "mean_macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(nested_cv_summary_df)


# ----------------------------------------------------------------
# Show fold-level outcomes and selected configurations
# ----------------------------------------------------------------
display(
    outer_results_df.sort_values(
        [
            "outer_fold",
            "method",
        ]
    )
)

display(
    selected_configurations_df.sort_values(
        "outer_fold"
    )
)


# ----------------------------------------------------------------
# Dataset-specific out-of-fold evaluation
# ----------------------------------------------------------------
dataset_oof_rows = []

for dataset_name, dataset_df in (
    nested_record_predictions_df.groupby(
        "dataset",
        sort=True,
    )
):
    metrics = calculate_macro_experiment_metrics(
        source_df=dataset_df,
        score_values=dataset_df[
            "hybrid_v3_risk_score"
        ].to_numpy(),
        prediction_values=dataset_df[
            "hybrid_v3_prediction"
        ].to_numpy(),
    )

    dataset_oof_rows.append({
        "dataset": dataset_name,
        "experiments": (
            dataset_df[
                "experiment_id"
            ].nunique()
        ),
        "records": len(
            dataset_df
        ),
        "macro_precision": (
            metrics[
                "macro_precision"
            ]
        ),
        "macro_recall": (
            metrics[
                "macro_recall"
            ]
        ),
        "macro_f1": (
            metrics[
                "macro_f1"
            ]
        ),
        "macro_fpr": (
            metrics[
                "macro_fpr"
            ]
        ),
        "macro_fnr": (
            metrics[
                "macro_fnr"
            ]
        ),
        "macro_balanced_accuracy": (
            metrics[
                "macro_balanced_accuracy"
            ]
        ),
        "macro_pr_auc": (
            metrics[
                "macro_pr_auc"
            ]
        ),
    })

dataset_oof_summary_df = pd.DataFrame(
    dataset_oof_rows
)

display(dataset_oof_summary_df)


# ----------------------------------------------------------------
# Diagnostics
# ----------------------------------------------------------------
v3_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "HYBRID_V3_NESTED"
        )
    ]
    .iloc[0]
)

ai_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "AI_CONSENSUS"
        )
    ]
    .iloc[0]
)

v1_row = (
    nested_cv_summary_df.loc[
        nested_cv_summary_df[
            "method"
        ].eq(
            "HYBRID_V1"
        )
    ]
    .iloc[0]
)

nested_diagnostics = {
    "v3_f1_above_v1": (
        v3_row[
            "mean_macro_f1"
        ]
        >
        v1_row[
            "mean_macro_f1"
        ]
    ),
    "v3_f1_above_ai_consensus": (
        v3_row[
            "mean_macro_f1"
        ]
        >
        ai_row[
            "mean_macro_f1"
        ]
    ),
    "v3_fpr_below_v1": (
        v3_row[
            "mean_macro_fpr"
        ]
        <
        v1_row[
            "mean_macro_fpr"
        ]
    ),
    "v3_fpr_within_ai_tolerance": (
        v3_row[
            "mean_macro_fpr"
        ]
        <= (
            ai_row[
                "mean_macro_fpr"
            ]
            + FPR_TOLERANCE_ABOVE_AI
        )
    ),
    "all_records_received_oof_prediction": (
        len(
            nested_record_predictions_df
        )
        == len(
            v3_source_df
        )
    ),
    "one_oof_prediction_per_record": (
        not nested_record_predictions_df
        .duplicated(
            subset=[
                "experiment_id",
                "dataset",
                "record_id",
            ]
        )
        .any()
    ),
}

nested_diagnostics_df = pd.DataFrame({
    "diagnostic": nested_diagnostics.keys(),
    "passed": nested_diagnostics.values(),
})

display(nested_diagnostics_df)


# ----------------------------------------------------------------
# Validation checks
# ----------------------------------------------------------------
validation_checks = {
    "three_outer_folds_completed": (
        outer_results_df[
            "outer_fold"
        ].nunique()
        == OUTER_FOLDS
    ),
    "three_methods_evaluated": (
        outer_results_df[
            "method"
        ].nunique()
        == 3
    ),
    "all_63_experiments_oof": (
        nested_record_predictions_df[
            "experiment_id"
        ].nunique()
        == 63
    ),
    "all_three_datasets_oof": (
        nested_record_predictions_df[
            "dataset"
        ].nunique()
        == 3
    ),
    "scores_in_valid_range": (
        nested_record_predictions_df[
            "hybrid_v3_risk_score"
        ]
        .between(
            0.0,
            1.0,
        )
        .all()
    ),
    "predictions_binary": (
        set(
            nested_record_predictions_df[
                "hybrid_v3_prediction"
            ]
            .unique()
        )
        .issubset(
            {0, 1}
        )
    ),
    "one_configuration_per_outer_fold": (
        len(
            selected_configurations_df
        )
        == OUTER_FOLDS
    ),
}

nested_validation_df = pd.DataFrame({
    "check": validation_checks.keys(),
    "passed": validation_checks.values(),
})

display(nested_validation_df)

assert nested_validation_df[
    "passed"
].all(), (
    "One or more nested-CV validation checks failed."
)


# ----------------------------------------------------------------
# Save artifacts
# ----------------------------------------------------------------
NESTED_OUTER_RESULTS_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_nested_outer_fold_results.csv"
)

NESTED_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_nested_cv_summary.csv"
)

NESTED_CONFIGURATIONS_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_selected_configurations.csv"
)

NESTED_INNER_SEARCH_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_inner_fold_search.csv"
)

NESTED_OOF_PREDICTIONS_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_out_of_fold_predictions.parquet"
)

NESTED_DATASET_SUMMARY_PATH = (
    HYBRID_RESULTS_DIR
    / "hybrid_v3_dataset_oof_summary.csv"
)

outer_results_df.to_csv(
    NESTED_OUTER_RESULTS_PATH,
    index=False,
)

nested_cv_summary_df.to_csv(
    NESTED_SUMMARY_PATH,
    index=False,
)

selected_configurations_df.to_csv(
    NESTED_CONFIGURATIONS_PATH,
    index=False,
)

inner_fold_search_df.to_csv(
    NESTED_INNER_SEARCH_PATH,
    index=False,
)

nested_record_predictions_df.to_parquet(
    NESTED_OOF_PREDICTIONS_PATH,
    index=False,
)

dataset_oof_summary_df.to_csv(
    NESTED_DATASET_SUMMARY_PATH,
    index=False,
)


# ----------------------------------------------------------------
# Save audit
# ----------------------------------------------------------------
cell_runtime_seconds = (
    time.perf_counter()
    - cell_started
)

nested_cv_audit = {
    "hybrid_run_id": HYBRID_RUN_ID,
    "version": V3_VERSION,
    "execution_timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "random_seed": RANDOM_SEED,
    "fpr_tolerance_above_ai": (
        FPR_TOLERANCE_ABOVE_AI
    ),
    "candidate_weight_count": len(
        CANDIDATE_WEIGHTS
    ),
    "candidate_threshold_count": len(
        CANDIDATE_THRESHOLDS
    ),
    "experiments": int(
        experiment_table_df[
            "experiment_id"
        ].nunique()
    ),
    "records": int(
        len(v3_source_df)
    ),
    "v3_mean_macro_f1": float(
        v3_row[
            "mean_macro_f1"
        ]
    ),
    "v3_mean_macro_fpr": float(
        v3_row[
            "mean_macro_fpr"
        ]
    ),
    "ai_mean_macro_f1": float(
        ai_row[
            "mean_macro_f1"
        ]
    ),
    "ai_mean_macro_fpr": float(
        ai_row[
            "mean_macro_fpr"
        ]
    ),
    "v1_mean_macro_f1": float(
        v1_row[
            "mean_macro_f1"
        ]
    ),
    "v1_mean_macro_fpr": float(
        v1_row[
            "mean_macro_fpr"
        ]
    ),
    "runtime_seconds": float(
        cell_runtime_seconds
    ),
    "summary_path": str(
        NESTED_SUMMARY_PATH
    ),
    "oof_predictions_path": str(
        NESTED_OOF_PREDICTIONS_PATH
    ),
}

NESTED_AUDIT_PATH = (
    HYBRID_AUDIT_DIR
    / "hybrid_v3_nested_cv_audit.json"
)

with open(
    NESTED_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as audit_file:
    json.dump(
        nested_cv_audit,
        audit_file,
        indent=2,
    )


# ----------------------------------------------------------------
# Completion summary
# ----------------------------------------------------------------
print("-" * 80)
print(
    "Nested constrained Hybrid V3 evaluation completed."
)
print(
    f"Version: {V3_VERSION}"
)
print(
    f"Outer folds: {OUTER_FOLDS}"
)
print(
    f"Inner folds: {INNER_FOLDS}"
)
print(
    f"Experiments evaluated out-of-fold: "
    f"{nested_record_predictions_df['experiment_id'].nunique()}"
)
print(
    f"V3 mean macro-F1: "
    f"{v3_row['mean_macro_f1']:.4f}"
)
print(
    f"V3 mean macro-FPR: "
    f"{v3_row['mean_macro_fpr']:.4f}"
)
print(
    f"AI-consensus mean macro-F1: "
    f"{ai_row['mean_macro_f1']:.4f}"
)
print(
    f"AI-consensus mean macro-FPR: "
    f"{ai_row['mean_macro_fpr']:.4f}"
)
print(
    f"Nested-CV summary: "
    f"{NESTED_SUMMARY_PATH}"
)
print(
    f"Out-of-fold predictions: "
    f"{NESTED_OOF_PREDICTIONS_PATH}"
)
print(
    f"Audit: "
    f"{NESTED_AUDIT_PATH}"
)
print(
    f"Runtime: "
    f"{cell_runtime_seconds:.2f} seconds"
)

## 7b. Governance Evaluation of the Selected Policy

This section applies the unified hierarchical routing policy to the out-of-fold evidence and reports the distribution, anomaly coverage, review burden, and purity of each governed action.


In [ ]:
import json
import time
import numpy as np
import pandas as pd

governance_started = time.perf_counter()

# ----------------------------------------------------------------
# Attach evidence columns required by the router
# ----------------------------------------------------------------
_evidence_columns = [
    'experiment_id',
    'dataset',
    'record_id',
    'rule_risk_score',
    'rule_severity_score',
    'rule_anomaly_prediction',
    'rule_evidence_available',
    'evidence_completeness',
    'evidence_uncertainty_score',
    'ai_consensus_score',
    'ai_prediction_vote',
    'ground_truth_label',
]

# Note: hybrid_decision_df contains the ground_truth_label
_missing = [c for c in _evidence_columns if c not in hybrid_decision_df.columns]
assert not _missing, f'Missing evidence columns for routing: {_missing}'

# Merge record predictions with evidence
# Drop redundant ground_truth_label from left to avoid suffixing
governed_df = nested_record_predictions_df.drop(columns=['ground_truth_label']).merge(
    hybrid_decision_df[_evidence_columns],
    on=['experiment_id', 'dataset', 'record_id'],
    how='left',
    validate='one_to_one',
)

# Apply the unified decision function
governed_df['governed_action'] = governed_df.apply(assign_hybrid_decision, axis=1)

# ACCEPT is the only non-intervention action.
governed_df['governed_prediction'] = (
    governed_df['governed_action'].ne('ACCEPT').astype(int)
)

# ----------------------------------------------------------------
# Action-level operational outcome
# ----------------------------------------------------------------
action_summary = (
    governed_df
    .groupby(['dataset', 'governed_action'], as_index=False)
    .agg(
        records=('record_id', 'size'),
        true_anomalies=('ground_truth_label', 'sum'),
        mean_ai_consensus=('ai_consensus_score', 'mean'),
        mean_rule_severity=('rule_severity_score', 'mean'),
        mean_uncertainty=('evidence_uncertainty_score', 'mean'),
    )
)
action_summary['anomaly_purity'] = (
    action_summary['true_anomalies'] / action_summary['records']
)
action_summary['action_rate'] = (
    action_summary['records']
    / action_summary.groupby('dataset')['records'].transform('sum')
)

action_summary.to_csv(
    HYBRID_RESULTS_DIR / 'governed_action_summary.csv', index=False
)

# ----------------------------------------------------------------
# Governed routing vs plain thresholding
# ----------------------------------------------------------------
def _binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {
        'precision': precision,
        'recall': recall,
        'f1': 2 * precision * recall / max(precision + recall, 1e-12),
        'fpr': fp / max(fp + tn, 1),
        'flagged': tp + fp,
        'false_reviews_per_tp': fp / max(tp, 1),
    }

comparison_rows = []
for experiment_id, group in governed_df.groupby('experiment_id'):
    for label, prediction_column in [
        ('HYBRID_V3_THRESHOLD', 'hybrid_v3_prediction'),
        ('HYBRID_V3_GOVERNED', 'governed_prediction'),
    ]:
        metrics = _binary_metrics(
            group['ground_truth_label'], group[prediction_column]
        )
        metrics.update({'experiment_id': experiment_id, 'method': label})
        comparison_rows.append(metrics)

governed_comparison_df = pd.DataFrame(comparison_rows)
governed_macro = (
    governed_comparison_df
    .groupby('method', as_index=False)[['precision', 'recall', 'f1', 'fpr',
                                        'false_reviews_per_tp']]
    .mean()
)
governed_macro.to_csv(
    HYBRID_RESULTS_DIR / 'governed_vs_threshold_comparison.csv', index=False
)

print('Governed action distribution by dataset:')
display(action_summary)

print('\nGoverned routing vs plain thresholding:')
display(governed_macro)

for dataset_name, group in governed_df.groupby('dataset'):
    actions = set(group['governed_action'].unique())
    assert 'QUARANTINE' in actions, f'{dataset_name}: QUARANTINE never fired.'

print('\nPASS: QUARANTINE is reachable in all experimental domains.')

In [ ]:
from sklearn.metrics import confusion_matrix

# ======================================================================
# Per-experiment metrics for the UNIFIED governed router.
# Everything downstream (paired tests, Friedman, win/loss, Fig 5,
# domain tables, operational burden) derives from this frame.
# ======================================================================

router_rows = []
for eid, grp in governed_df.groupby("experiment_id"):
    y = grp["ground_truth_label"].to_numpy(int)
    p = grp["governed_prediction"].to_numpy(int)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    assert (tn + fp) > 0, f"{eid}: no negative records"
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    router_rows.append({
        "experiment_id": eid,
        "dataset": grp["dataset"].iloc[0],
        "method": "HYBRID_V3_GOVERNED",
        "true_positive": int(tp), "false_positive": int(fp),
        "false_negative": int(fn), "true_negative": int(tn),
        "precision": prec, "recall": rec,
        "f1": 2 * prec * rec / max(prec + rec, 1e-12),
        "false_positive_rate": fp / max(fp + tn, 1),
        "false_negative_rate": fn / max(fn + tp, 1),
        "specificity": spec,
        "balanced_accuracy": (rec + spec) / 2,
    })

router_metrics_df = pd.DataFrame(router_rows).merge(
    experiment_registry_df[["experiment_id", "anomaly_type", "anomaly_rate"]],
    on="experiment_id", how="left")

assert len(router_metrics_df) == 63, f"expected 63 rows, got {len(router_metrics_df)}"

# Append to the frame the statistical cells consume
statistical_experiment_metrics_df = pd.concat(
    [statistical_experiment_metrics_df[
         statistical_experiment_metrics_df["method"] != "HYBRID_V3_GOVERNED"],
     router_metrics_df], ignore_index=True)
statistical_experiment_metrics_df.to_csv(
    HYBRID_RESULTS_DIR / "statistical_experiment_metrics.csv", index=False)

# ----------------------------------------------------------------
# Operational burden — REBUILT FROM SCRATCH for every method.
#
# This previously read:
#     operational_burden_df = pd.concat(
#         [operational_burden_df[... != "HYBRID_V3_GOVERNED"], router_burden])
#
# No cell in this notebook ever CREATED operational_burden_df. It was a stale
# global left in memory from an earlier session, and only the router row was
# ever replaced. Every other row — including HYBRID_V1 — was a fossil, which
# is why V1's numbers stayed byte-identical across runs no matter what was
# fixed upstream.
#
# The table is now derived from statistical_experiment_metrics_df, the same
# frame the paired tests and Friedman tests use, so burden and statistics
# cannot disagree.
# ----------------------------------------------------------------
_burden_source = statistical_experiment_metrics_df.copy()

# The frame carries two naming conventions depending on which cell wrote the
# rows. Coalesce them before aggregating.
for _short, _long in [("tp", "true_positive"), ("fp", "false_positive"),
                      ("fn", "false_negative"), ("tn", "true_negative")]:
    if _short in _burden_source.columns and _long in _burden_source.columns:
        _burden_source[_short] = _burden_source[_short].fillna(
            _burden_source[_long])
    elif _long in _burden_source.columns:
        _burden_source[_short] = _burden_source[_long]
    assert _short in _burden_source.columns, \
        f"confusion-matrix column {_short} missing from metrics frame"
    assert _burden_source[_short].notna().all(), \
        f"confusion-matrix column {_short} has missing values"

_agg = (_burden_source.groupby("method", as_index=False)[["tp", "fp", "fn", "tn"]]
        .sum())

_agg["total_records"]        = _agg[["tp", "fp", "fn", "tn"]].sum(axis=1).astype(int)
_agg["total_true_anomalies"] = (_agg["tp"] + _agg["fn"]).astype(int)
_agg["flagged_records"]      = (_agg["tp"] + _agg["fp"]).astype(int)
_agg["flagged_record_rate"]  = _agg["flagged_records"] / _agg["total_records"]
_agg["true_positive_records"]  = _agg["tp"].astype(int)
_agg["false_positive_records"] = _agg["fp"].astype(int)
_agg["missed_anomalies"]       = _agg["fn"].astype(int)
_agg["anomalies_captured_rate"] = _agg["tp"] / _agg["total_true_anomalies"].clip(lower=1)
_agg["false_reviews_per_true_positive"]   = _agg["fp"] / _agg["tp"].clip(lower=1)
_agg["records_flagged_per_true_positive"] = _agg["flagged_records"] / _agg["tp"].clip(lower=1)

operational_burden_df = _agg[[
    "method", "total_records", "total_true_anomalies", "flagged_records",
    "flagged_record_rate", "true_positive_records", "false_positive_records",
    "missed_anomalies", "anomalies_captured_rate",
    "false_reviews_per_true_positive", "records_flagged_per_true_positive",
]].copy()

# Every method must describe the same corpus.
_record_totals = operational_burden_df["total_records"].unique()
assert len(_record_totals) == 1, \
    f"methods cover different record counts: {_record_totals}"
_anomaly_totals = operational_burden_df["total_true_anomalies"].unique()
assert len(_anomaly_totals) == 1, \
    f"methods see different anomaly counts: {_anomaly_totals}"

operational_burden_df.to_csv(
    HYBRID_RESULTS_DIR / "operational_intervention_burden.csv", index=False)

print("\nOperational burden rebuilt for all methods:")
print(operational_burden_df[["method", "flagged_records", "flagged_record_rate",
                             "true_positive_records",
                             "false_reviews_per_true_positive"]]
      .sort_values("flagged_records").to_string(index=False))

# Domain-level macro summary
router_domain_df = (router_metrics_df.groupby("dataset", as_index=False)
    .agg(macro_precision=("precision", "mean"), macro_recall=("recall", "mean"),
         macro_f1=("f1", "mean"), macro_fpr=("false_positive_rate", "mean"),
         macro_fnr=("false_negative_rate", "mean"),
         macro_balanced_accuracy=("balanced_accuracy", "mean")))
router_domain_df["method"] = "HYBRID_V3_GOVERNED"
router_domain_df.to_csv(
    HYBRID_RESULTS_DIR / "governed_router_dataset_summary.csv", index=False)

print("Unified Router Macro Means:")
print(router_metrics_df[["precision","recall","f1","false_positive_rate"]].mean())
display(router_domain_df)

## 8. Statistical and Operational Analysis

This section performs paired statistical comparisons, multiple-comparison correction, omnibus testing, and operational burden analysis.


In [ ]:
# ======================================================================
# Section 8 — Statistical and operational analysis
#
# All statistical outputs are recomputed from the current validated
# experiment-level metrics and written to versioned artifacts.
# ======================================================================

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon

cell_started = time.perf_counter()

STATISTICAL_ANALYSIS_VERSION = "HYBRID_STATISTICAL_ANALYSIS_V2.0"
BOOTSTRAP_ITERATIONS  = 10_000
BOOTSTRAP_RANDOM_SEED = 42
SIGNIFICANCE_LEVEL    = 0.05

FRIEDMAN_METHODS = [
    "RULE_ONLY", "AI_CONSENSUS", "HYBRID_V1",
    "HYBRID_V3_NESTED", "HYBRID_V3_GOVERNED",
]

PAIRWISE_COMPARISONS = [
    ("HYBRID_V3_GOVERNED", "AI_CONSENSUS"),
    ("HYBRID_V3_GOVERNED", "RULE_ONLY"),
    ("HYBRID_V3_GOVERNED", "HYBRID_V1"),
    ("HYBRID_V3_GOVERNED", "HYBRID_V3_NESTED"),
    ("HYBRID_V3_NESTED",   "AI_CONSENSUS"),
    ("HYBRID_V3_NESTED",   "HYBRID_V1"),
]

METRIC_NAMES = ["precision", "recall", "f1",
                "false_positive_rate", "false_negative_rate", "balanced_accuracy"]


# ----------------------------------------------------------------
# Guards
# ----------------------------------------------------------------
_present = set(statistical_experiment_metrics_df["method"].unique())
_missing = set(FRIEDMAN_METHODS) - _present
assert not _missing, (
    f"Methods absent from statistical_experiment_metrics_df: {sorted(_missing)}. "
    f"Present: {sorted(_present)}. Re-run Section 7b."
)

pivots = {
    metric: statistical_experiment_metrics_df.pivot(
        index="experiment_id", columns="method", values=metric)
    for metric in METRIC_NAMES
}
for metric, table in pivots.items():
    assert len(table) == 63, f"{metric}: expected 63 experiments, got {len(table)}"
    assert not table[FRIEDMAN_METHODS].isna().any().any(), \
        f"{metric}: missing values for one or more methods"

rng = np.random.default_rng(BOOTSTRAP_RANDOM_SEED)


def paired_bootstrap(differences, iterations=BOOTSTRAP_ITERATIONS):
    """Percentile CI on the mean paired difference."""
    idx = rng.integers(0, len(differences), size=(iterations, len(differences)))
    means = differences[idx].mean(axis=1)
    return (float(np.percentile(means, 2.5)),
            float(np.percentile(means, 97.5)),
            float((means > 0).mean()))


def holm_adjust(p_values):
    """Holm step-down correction with monotonicity enforced."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, position in enumerate(order):
        value = min(1.0, (len(p_values) - rank) * p_values[position])
        running = max(running, value)
        adjusted[position] = running
    return adjusted


# ----------------------------------------------------------------
# Paired comparisons
# ----------------------------------------------------------------
rows = []
for metric in METRIC_NAMES:
    table = pivots[metric]
    for method_a, method_b in PAIRWISE_COMPARISONS:
        differences = (table[method_a] - table[method_b]).to_numpy(float)
        nonzero = differences[differences != 0]
        ci_low, ci_high, prob_above = paired_bootstrap(differences)

        if len(nonzero) == 0:
            statistic, raw_p = np.nan, 1.0
        else:
            statistic, raw_p = wilcoxon(nonzero)

        rows.append({
            "metric": metric, "method_a": method_a, "method_b": method_b,
            "experiments": len(differences),
            "method_a_mean": float(table[method_a].mean()),
            "method_b_mean": float(table[method_b].mean()),
            "mean_difference": float(differences.mean()),
            "ci_95_lower": ci_low, "ci_95_upper": ci_high,
            "bootstrap_probability_above_zero": prob_above,
            "wilcoxon_statistic": statistic, "wilcoxon_p_value": float(raw_p),
            "nonzero_pairs": int(len(nonzero)),
        })

paired_statistical_tests_df = pd.DataFrame(rows)
paired_statistical_tests_df["holm_adjusted_p_value"] = holm_adjust(
    paired_statistical_tests_df["wilcoxon_p_value"])
paired_statistical_tests_df["statistically_significant"] = (
    paired_statistical_tests_df["holm_adjusted_p_value"] < SIGNIFICANCE_LEVEL)
paired_statistical_tests_df.to_csv(
    HYBRID_RESULTS_DIR / "paired_statistical_tests.csv", index=False)


# ----------------------------------------------------------------
# Friedman omnibus tests
# ----------------------------------------------------------------
friedman_rows = []
for metric in METRIC_NAMES:
    table = pivots[metric]
    statistic, p_value = friedmanchisquare(
        *[table[m].to_numpy(float) for m in FRIEDMAN_METHODS])
    friedman_rows.append({
        "metric": metric,
        "methods_compared": len(FRIEDMAN_METHODS),
        "degrees_of_freedom": len(FRIEDMAN_METHODS) - 1,
        "experiments": len(table),
        "friedman_statistic": float(statistic),
        "friedman_p_value": float(p_value),
        "statistically_significant": bool(p_value < SIGNIFICANCE_LEVEL),
    })
friedman_omnibus_df = pd.DataFrame(friedman_rows)
friedman_omnibus_df.to_csv(
    HYBRID_RESULTS_DIR / "friedman_omnibus_tests.csv", index=False)


# ----------------------------------------------------------------
# Experiment-level win / loss
# ----------------------------------------------------------------
f1_table  = pivots["f1"]
fpr_table = pivots["false_positive_rate"]
win_loss_rows = []
for method_a, method_b in PAIRWISE_COMPARISONS:
    f1_diff  = f1_table[method_a]  - f1_table[method_b]
    fpr_diff = fpr_table[method_a] - fpr_table[method_b]
    win_loss_rows.append({
        "comparison": f"{method_a} vs {method_b}",
        "f1_wins": int((f1_diff > 0).sum()),
        "f1_ties": int((f1_diff == 0).sum()),
        "f1_losses": int((f1_diff < 0).sum()),
        "lower_fpr_wins": int((fpr_diff < 0).sum()),
        "equal_fpr_ties": int((fpr_diff == 0).sum()),
        "higher_fpr_losses": int((fpr_diff > 0).sum()),
        "mean_f1_difference": float(f1_diff.mean()),
        "mean_fpr_difference": float(fpr_diff.mean()),
    })
experiment_win_loss_df = pd.DataFrame(win_loss_rows)
experiment_win_loss_df.to_csv(
    HYBRID_RESULTS_DIR / "experiment_win_loss_analysis.csv", index=False)


# ----------------------------------------------------------------
# Per-domain paired differences
# ----------------------------------------------------------------
domain_rows = []
for dataset_name, group in statistical_experiment_metrics_df.groupby("dataset"):
    f1_pivot  = group.pivot(index="experiment_id", columns="method", values="f1")
    fpr_pivot = group.pivot(index="experiment_id", columns="method",
                            values="false_positive_rate")
    for method_a, method_b in PAIRWISE_COMPARISONS:
        if method_a not in f1_pivot or method_b not in f1_pivot:
            continue
        domain_rows.append({
            "dataset": dataset_name, "method_a": method_a,
            "comparison_method": method_b, "experiments": len(f1_pivot),
            "mean_f1_difference": float((f1_pivot[method_a] - f1_pivot[method_b]).mean()),
            "mean_fpr_difference": float((fpr_pivot[method_a] - fpr_pivot[method_b]).mean()),
        })
dataset_paired_differences_df = pd.DataFrame(domain_rows)
dataset_paired_differences_df.to_csv(
    HYBRID_RESULTS_DIR / "dataset_paired_differences.csv", index=False)


# ----------------------------------------------------------------
# Audit record
# ----------------------------------------------------------------
best_method = pivots["f1"][FRIEDMAN_METHODS].mean().idxmax()

with open(HYBRID_AUDIT_DIR / "statistical_analysis_audit.json", "w") as handle:
    json.dump({
        "version": STATISTICAL_ANALYSIS_VERSION,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        "bootstrap_seed": BOOTSTRAP_RANDOM_SEED,
        "significance_level": SIGNIFICANCE_LEVEL,
        "holm_family_size": int(len(paired_statistical_tests_df)),
        "friedman_methods": FRIEDMAN_METHODS,
        "pairwise_comparisons": [f"{a} vs {b}" for a, b in PAIRWISE_COMPARISONS],
        "best_method_by_mean_macro_f1": best_method,
        "runtime_seconds": time.perf_counter() - cell_started,
    }, handle, indent=2)


# ----------------------------------------------------------------
# Cross-check every method label against the artifact that defines it.
#
# Three cells independently map method names to prediction columns, and
# nothing previously verified they agreed. Two label collisions went
# undetected for several runs as a result.
# ----------------------------------------------------------------
_v1_summary_path = HYBRID_RESULTS_DIR / "v1_prototype_decision_summary.csv"
if _v1_summary_path.exists() and "operational_burden_df" in globals():
    _v1_summary = pd.read_csv(_v1_summary_path)
    _expected_v1_flagged = int(
        _v1_summary.loc[_v1_summary["v1_prototype_decision"].ne("ACCEPT"),
                        "records"].sum())
    _v1_burden = operational_burden_df.loc[
        operational_burden_df["method"].eq("HYBRID_V1")]
    if not _v1_burden.empty:
        _actual_v1_flagged = int(_v1_burden["flagged_records"].iloc[0])
        assert _expected_v1_flagged == _actual_v1_flagged, (
            f"HYBRID_V1 flags {_actual_v1_flagged:,} records but the frozen "
            f"prototype flags {_expected_v1_flagged:,}. HYBRID_V1 is not the "
            f"frozen policy — check Fix A (cell 7) and Fix B (cell 11)."
        )
        print(f"PASS: HYBRID_V1 matches the frozen prototype "
              f"({_expected_v1_flagged:,} flagged)\n")

_router_burden = operational_burden_df.loc[
    operational_burden_df["method"].eq("HYBRID_V3_GOVERNED")] \
    if "operational_burden_df" in globals() else None
if _router_burden is not None and not _router_burden.empty:
    _expected_router_flagged = int(
        (governed_df["governed_prediction"] == 1).sum())
    _actual_router_flagged = int(_router_burden["flagged_records"].iloc[0])
    assert _expected_router_flagged == _actual_router_flagged, (
        f"HYBRID_V3_GOVERNED flags {_actual_router_flagged:,} but governed_df "
        f"flags {_expected_router_flagged:,}."
    )
    print(f"PASS: HYBRID_V3_GOVERNED matches governed_df "
          f"({_expected_router_flagged:,} flagged)\n")


print(f"Paired tests: {len(paired_statistical_tests_df)} "
      f"(Holm family size {len(paired_statistical_tests_df)})")
print(f"Best method by mean macro-F1: {best_method}")
print(f"Runtime: {time.perf_counter() - cell_started:.1f}s\n")

print("Governed router comparisons:")
display(
    paired_statistical_tests_df
    .loc[paired_statistical_tests_df["method_a"].eq("HYBRID_V3_GOVERNED"),
         ["metric", "method_b", "mean_difference", "ci_95_lower", "ci_95_upper",
          "holm_adjusted_p_value", "statistically_significant"]]
    .round(4)
)

print("\nFriedman omnibus tests:")
display(friedman_omnibus_df.round(4))

print("\nWin / loss:")
display(experiment_win_loss_df)


## 9. Publication Artifact Generation

This section exports machine-readable tables, statistical summaries, and publication-quality figures directly into repository folders.


In [ ]:
# ======================================================================
# Section 9 — Publication artifacts: Figures 1 to 3
#
# Figures 1 to 3 are generated directly from the validated publication
# tables so that the notebook reproduces every plotted value.
# ======================================================================

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cell_started = time.perf_counter()
PUBLICATION_ARTIFACT_VERSION = "HYBRID_PUBLICATION_ARTIFACTS_V2.0"

METHOD_LABELS = {
    "RULE_ONLY": "Rule-only",
    "ISOLATION_FOREST": "Isolation Forest",
    "LOCAL_OUTLIER_FACTOR": "Local Outlier Factor",
    "AI_CONSENSUS": "Detector consensus",
    "HYBRID_V1": "Hybrid V1 (uncalibrated)",
    "HYBRID_V3_NESTED": "Hybrid V3 (detection component)",
    "HYBRID_V3_GOVERNED": "Governed router",
}


# ----------------------------------------------------------------
# Assemble every method onto one frame
# ----------------------------------------------------------------
publication_summary_aligned = publication_summary_df.rename(columns={
    "macro_f1": "mean_macro_f1", "macro_fpr": "mean_macro_fpr"})

router_point = pd.DataFrame([{
    "method": "HYBRID_V3_GOVERNED",
    "mean_macro_f1":  float(router_metrics_df["f1"].mean()),
    "mean_macro_fpr": float(router_metrics_df["false_positive_rate"].mean()),
}])

overall_df = pd.concat([
    nested_cv_summary_df,
    publication_summary_aligned[
        ~publication_summary_aligned["method"].isin(nested_cv_summary_df["method"])],
    router_point,
# Retain the current governed-router result when resolving duplicate labels.ED row from
# publication_summary_df win over router_point, which is appended last.
# Figures 1 and 3 therefore plotted a different router than the statistics used.
], ignore_index=True).drop_duplicates(subset="method", keep="last")

overall_df["Method Label"] = overall_df["method"].map(METHOD_LABELS)

_unmapped = overall_df.loc[overall_df["Method Label"].isna(), "method"].tolist()
assert not _unmapped, f"Unmapped method codes: {_unmapped}"
_null = overall_df.loc[
    overall_df["mean_macro_f1"].isna() | overall_df["mean_macro_fpr"].isna(),
    "method"].tolist()
assert not _null, f"Missing F1/FPR after merge for: {_null}"
assert set(METHOD_LABELS) <= set(overall_df["method"]), \
    f"Methods absent: {set(METHOD_LABELS) - set(overall_df['method'])}"

# The figures must agree with the frame the statistics are computed on.
_router_figure_f1 = float(
    overall_df.loc[overall_df["method"].eq("HYBRID_V3_GOVERNED"),
                   "mean_macro_f1"].iloc[0])
_router_metrics_f1 = float(router_metrics_df["f1"].mean())
assert abs(_router_figure_f1 - _router_metrics_f1) < 1e-6, (
    f"Figure router macro-F1 {_router_figure_f1:.4f} disagrees with "
    f"router_metrics_df {_router_metrics_f1:.4f}"
)

_router_figure_fpr = float(
    overall_df.loc[overall_df["method"].eq("HYBRID_V3_GOVERNED"),
                   "mean_macro_fpr"].iloc[0])
_router_metrics_fpr = float(router_metrics_df["false_positive_rate"].mean())
assert abs(_router_figure_fpr - _router_metrics_fpr) < 1e-6, (
    f"Figure router macro-FPR {_router_figure_fpr:.4f} disagrees with "
    f"router_metrics_df {_router_metrics_fpr:.4f}"
)

overall_df = overall_df.sort_values("mean_macro_f1", ascending=False)


# ----------------------------------------------------------------
# Figure 1 — overall macro-F1 and FPR, all methods
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

sns.barplot(x="Method Label", y="mean_macro_f1", data=overall_df,
            ax=axes[0], palette="viridis")
axes[0].set_title("Macro-F1 across 63 experiments")
axes[0].set_ylabel("Macro-F1")
axes[0].set_ylim(0, max(0.45, overall_df["mean_macro_f1"].max() * 1.15))

sns.barplot(x="Method Label", y="mean_macro_fpr", data=overall_df,
            ax=axes[1], palette="magma")
axes[1].set_title("Macro false-positive rate")
axes[1].set_ylabel("False-positive rate")
axes[1].set_ylim(0, max(0.20, overall_df["mean_macro_fpr"].max() * 1.15))

for ax in axes:
    ax.set_xlabel("Method")
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_ha("right")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure_1_nested_cv_performance.png", dpi=300)
plt.show()


# ----------------------------------------------------------------
# Figure 2 — domain-specific macro-F1, all methods
# ----------------------------------------------------------------
domain_frames = []
if "dataset_publication_summary_df" in globals():
    domain_frames.append(dataset_publication_summary_df.copy())
domain_frames.append(router_domain_df.copy())

domain_df = pd.concat(domain_frames, ignore_index=True)
domain_df = domain_df.drop_duplicates(subset=["dataset", "method"], keep="last")
domain_df["Method Label"] = domain_df["method"].map(METHOD_LABELS)
domain_df["Dataset"] = domain_df["dataset"].str.capitalize()

_unmapped_domain = domain_df.loc[domain_df["Method Label"].isna(), "method"].tolist()
assert not _unmapped_domain, f"Unmapped in domain frame: {_unmapped_domain}"
assert "HYBRID_V3_GOVERNED" in set(domain_df["method"]), \
    "Governed router missing from the domain frame"

plt.figure(figsize=(12, 6))
ax = sns.barplot(x="Dataset", y="macro_f1", hue="Method Label", data=domain_df)
ax.set_title("Domain-specific macro-F1")
ax.set_ylabel("Macro-F1")
ax.set_xlabel("Domain")
ax.set_ylim(0, max(0.60, domain_df["macro_f1"].max() * 1.15))
ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure_2_domain_specific_performance.png", dpi=300)
plt.show()


# ----------------------------------------------------------------
# Figure 3 — F1 against FPR, with a computed Pareto frontier
# ----------------------------------------------------------------
def _is_pareto(row, frame):
    """Maximise F1, minimise FPR."""
    others = frame[frame["method"] != row["method"]]
    dominated = (
        (others["mean_macro_f1"] >= row["mean_macro_f1"])
        & (others["mean_macro_fpr"] <= row["mean_macro_fpr"])
        & ((others["mean_macro_f1"] > row["mean_macro_f1"])
           | (others["mean_macro_fpr"] < row["mean_macro_fpr"]))
    )
    return not dominated.any()

overall_df["pareto"] = overall_df.apply(lambda r: _is_pareto(r, overall_df), axis=1)
print("Pareto-optimal methods:",
      overall_df.loc[overall_df["pareto"], "Method Label"].tolist())

fig, ax = plt.subplots(figsize=(9.5, 6.5))

frontier = overall_df[overall_df["pareto"]].sort_values("mean_macro_fpr")
ax.plot(frontier["mean_macro_fpr"], frontier["mean_macro_f1"],
        linestyle="--", linewidth=1.2, color="grey", alpha=0.75,
        zorder=1, label="Pareto frontier")

for _, row in overall_df.iterrows():
    ax.scatter(row["mean_macro_fpr"], row["mean_macro_f1"],
               s=170 if row["pareto"] else 110,
               edgecolor="black" if row["pareto"] else "none",
               linewidth=1.3 if row["pareto"] else 0, zorder=3)
    ax.annotate(row["Method Label"],
                (row["mean_macro_fpr"], row["mean_macro_f1"]),
                textcoords="offset points", xytext=(10, 6), fontsize=9,
                fontweight="bold" if row["pareto"] else "normal")

ax.set_xlabel("Macro false-positive rate")
ax.set_ylabel("Macro-F1")
ax.set_ylim(0, max(0.45, overall_df["mean_macro_f1"].max() * 1.20))
ax.set_title("Detection performance against false-positive burden")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure_3_operational_tradeoff.png", dpi=300)
plt.show()


# ----------------------------------------------------------------
# Publication table
# ----------------------------------------------------------------
publication_table = overall_df[
    ["Method Label", "mean_macro_f1", "mean_macro_fpr", "pareto"]].copy()
publication_table.columns = ["Method", "Macro-F1", "Macro-FPR", "Pareto optimal"]
publication_table.to_csv(
    TABLES_DIR / "table_1_publication_performance.csv", index=False)

print(f"\nSection 9 runtime: {time.perf_counter() - cell_started:.1f}s")
display(publication_table.round(4))


In [ ]:
# Figure 4: Operational Efficiency (False Reviews per True Positive)
plt.figure(figsize=(10, 6))
fig4_data = operational_burden_df.copy()
fig4_data['Method Label'] = fig4_data['method'].map(METHOD_LABELS)

sns.barplot(x='Method Label', y='false_reviews_per_true_positive', data=fig4_data, palette='rocket')
plt.title('Human-in-the-Loop Burden (Governed)')
plt.ylabel('False Reviews per True Positive')
plt.xlabel('Methodology')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure_4_operational_burden.png', dpi=300)
plt.show()

In [ ]:
# Figure 5: Statistical Stability (F1 Distribution across 63 folds)
plt.figure(figsize=(12, 6))
fig5_data = statistical_experiment_metrics_df.copy()
fig5_data['Method Label'] = fig5_data['method'].map(METHOD_LABELS)

sns.boxplot(x='Method Label', y='f1', data=fig5_data, palette='Set2', showfliers=False)
sns.stripplot(x='Method Label', y='f1', data=fig5_data, color='black', alpha=0.3, jitter=True)
plt.title('Macro-F1 Distribution (Governed Taxonomy)')
plt.ylabel('Macro-F1 Score')
plt.xlabel('Methodology')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure_5_f1_distribution.png', dpi=300)
plt.show()

## 10. Reproducibility Completion Report

The following paths contain the final publication and verification artifacts generated by this notebook.

In [ ]:
completion_report = {
    "selected_policy": "Hybrid V3 / W08 when selected by the nested procedure",
    "figures_directory": FIGURES_DIR,
    "tables_directory": TABLES_DIR,
    "results_directory": RESULTS_DIR,
    "configuration_directory": HYBRID_CONFIG_DIR,
    "audit_directory": HYBRID_AUDIT_DIR,
}

for item, value in completion_report.items():
    print(f"{item}: {value}")

## 11. Artifact Archival, Staleness Check, and Manifest

This section validates that publication artifacts were generated by the current run, removes superseded naming variants, and writes a checksummed manifest for reproducibility and archival deposit.


In [ ]:
# ======================================================================
# Section 11 — Archive stale artifacts, verify freshness, write manifest
# ======================================================================

import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ARCHIVE_ROOT = PROJECT_ROOT / "_archive"
ARCHIVE_ROOT.mkdir(exist_ok=True)

STALE_TOLERANCE_HOURS = 6

CURRENT_FIGURES = {
    "figure_1_nested_cv_performance.png",
    "figure_2_domain_specific_performance.png",
    "figure_3_operational_tradeoff.png",
    "figure_4_operational_burden.png",
    "figure_5_f1_distribution.png",
}


# ----------------------------------------------------------------
# 1. Retire the superseded July result tree
# ----------------------------------------------------------------
legacy_tree = PROJECT_ROOT / "hybrid_decision_engine"
if legacy_tree.exists() and legacy_tree.is_dir():
    destination = ARCHIVE_ROOT / "hybrid_decision_engine_2026-07-27"
    if destination.exists():
        print(f"Legacy tree already archived at {destination}")
    else:
        shutil.move(str(legacy_tree), str(destination))
        print(f"Archived legacy tree -> {destination}")
else:
    print("No legacy hybrid_decision_engine/ at project root.")


# ----------------------------------------------------------------
# 2. Retire superseded figure filenames
# ----------------------------------------------------------------
figure_archive = ARCHIVE_ROOT / f"figures_{RUN_STAMP}"
archived_figures = 0
for path in sorted(FIGURES_DIR.glob("*.png")):
    if path.name not in CURRENT_FIGURES:
        figure_archive.mkdir(parents=True, exist_ok=True)
        shutil.move(str(path), str(figure_archive / path.name))
        print(f"Archived stale figure: {path.name}")
        archived_figures += 1
if archived_figures == 0:
    print("No stale figure filenames found.")

for expected in sorted(CURRENT_FIGURES):
    assert (FIGURES_DIR / expected).exists(), f"Missing current figure: {expected}"


# ----------------------------------------------------------------
# 3. Staleness check
# ----------------------------------------------------------------
now = datetime.now(timezone.utc).timestamp()
stale = []
for folder in [HYBRID_RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            age_hours = (now - path.stat().st_mtime) / 3600
            if age_hours > STALE_TOLERANCE_HOURS:
                stale.append((str(path.relative_to(PROJECT_ROOT)), round(age_hours, 1)))

if stale:
    print(f"\nWARNING — {len(stale)} artifact(s) older than "
          f"{STALE_TOLERANCE_HOURS}h, i.e. NOT regenerated by this run:")
    for name, age in sorted(stale, key=lambda item: -item[1]):
        print(f"  {name:60} {age:7.1f} h")
    print("Re-run the cell that produces each before using these numbers.")
else:
    print(f"\nPASS: every artifact was written within the last "
          f"{STALE_TOLERANCE_HOURS} hours.")


# ----------------------------------------------------------------
# 4. Checksummed manifest
# ----------------------------------------------------------------
def sha256_of(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


entries = []
for folder in [HYBRID_RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
               HYBRID_CONFIG_DIR, HYBRID_AUDIT_DIR]:
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            entries.append({
                "path": str(path.relative_to(PROJECT_ROOT)),
                "bytes": path.stat().st_size,
                "sha256": sha256_of(path),
                "modified_utc": datetime.fromtimestamp(
                    path.stat().st_mtime, timezone.utc).isoformat(),
            })

manifest = {
    "artifact_version": "HYBRID_PUBLICATION_ARTIFACTS_V2.0",
    "statistical_analysis_version": STATISTICAL_ANALYSIS_VERSION,
    "policy_version": ROUTING_POLICY["policy_version"],
    "run_stamp": RUN_STAMP,
    "hybrid_run_id": HYBRID_RUN_ID,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "file_count": len(entries),
    "stale_files": stale,
    "files": entries,
}

manifest_path = PROJECT_ROOT / f"publication_manifest_{RUN_STAMP}.json"
with open(manifest_path, "w") as handle:
    json.dump(manifest, handle, indent=2)

print(f"\nManifest written: {len(entries)} files -> {manifest_path.name}")
print("Archival complete.")
